# z-shift Tier B - GPU experiments (trimmed: B2 + B4)

The full protocol defines eight CUDA experiments, **B1-B8**; this notebook runs
only **B2** (reconstruction accuracy baseline) and **B4** (frame-budget
ablation) — picked as the minimum a 6-page paper's results section needs: B2
closes the paper's own admitted gap ("no standardized geometric metrics
reported"), and B4 is the one experiment that can surface a real bug
(motion-rank frame ordering degrading `swin` pairing) rather than just confirm
an expected behaviour. The other six `exp_b*.py` modules still exist in
`bench/`; re-add their entries to `EXPERIMENTS`, `MODULES`, `SMOKE_ARGS`/
`FULL_ARGS` and section 14's run cells to bring one back. Tier A is CPU-only
and is not run here.

Runs on **Kaggle** (recommended) or **Colab**. Kaggle is preferred because
*Save & Run All* executes headless for up to 12 h and survives a closed browser,
and because a private Kaggle Dataset mounts your DTU / capture data read-only at
`/kaggle/input` every session with no re-download.

## Read this before you press Run All

1. **`--quick` is a no-op in Tier B.** `experiment_parser` defines the flag and
   not one of the eight `exp_b*.py` modules forwards it to `run()`. Typing the
   smoke-test command in a terminal silently launches the full grid. This
   notebook therefore never passes `--quick`; scope is controlled by the flags
   that are actually honoured (`--scenes`, `--n-images`, `--frame-counts`,
   `--thresholds`, `--no-budget-sweep`). Wiring `--quick` properly is still
   TODO P0 and worth doing.
2. **Run the SMOKE lane first.** It exercises both modules end to end in
   minutes. A three-day run that dies at row 1 on a `KeyError` is the failure
   mode this exists to prevent.
3. **The repo is cloned from GitHub.** `bench/`, `paper/`,
   `instrumentation.py` and `tests/test_bench.py` are untracked on your laptop.
   Commit and push before running this, or the clone will not contain them.
4. **`data/` and `third_party/` are gitignored.** MASt3R is cloned and installed
   by this notebook; the dataset must come from a mounted Kaggle Dataset or
   Drive folder that you point `DATA_ROOT` at.

5. **The dataset test (section 10) runs before anything expensive.** It reads
   the Kaggle datasets you actually mounted rather than a hardcoded slug, and
   fails in seconds on what would otherwise fail three hours in: an unreadable
   image, a ground-truth file trimesh cannot open, a `tau` stated in the wrong
   unit, a scene with no ground truth at all.

## What "proper error logs" means here

- One log file per experiment under `<results>/logs/<run_id>/`, plus a combined
  `session.log` holding every log record from the notebook itself.
- Each experiment runs in its **own subprocess**, so a CUDA OOM or a VTK
  segfault kills that experiment and not the kernel. Exit code, signal, wall
  time and the tail of the output are recorded either way.
- `faulthandler` is enabled in parent and children, so a native crash leaves a
  Python traceback instead of a silent death.
- `run_summary.json` is rewritten after **every** experiment, so a session
  killed at the 12 h wall still leaves a full account of what finished.
- Re-running the notebook **skips experiments whose CSV already exists**, so a
  killed session resumes rather than restarts.

## How to run this, and what it costs

**Kaggle is the right platform for the full lane.** *Save & Run All* executes
headless for up to 12 h and survives a closed browser; Colab free disconnects on
idle after ~90 minutes and needs the tab open, which makes it a smoke-lane tool.

### Kaggle

1. **New Notebook > File > Import Notebook**, upload `notebooks/tier_b_gpu.ipynb`.
2. **Settings > Accelerator > GPU T4 x2** (or P100), **Internet > On**. Internet
   is not optional: `pip`, the MASt3R clone and the 2.6 GB checkpoint all need it.
3. **Attaching a dataset is optional.** DTU ships images and ground truth as
   separate archives and its server honours byte ranges, so section 8 reads
   only what the run needs out of them: 0.57 GB of images for the five scans in
   `DTU_SCANS` at one lighting condition, out of a 129.6 GB archive, and 0.61 GB
   of reference clouds (88–140 MB a scan) out of 6.97 GB. That needs
   *Internet > On*, which this notebook already requires for pip and the
   checkpoint. `/kaggle/temp` is wiped between sessions, so a multi-session
   lane re-fetches it each time — worth turning the first fetch into a private
   Kaggle Dataset and pointing `DTU_IMAGES_SLUG` at that, which is then a
   mirror you have verified yourself.

   Attach a mirror if you have one — it mounts instantly and downloads nothing
   — and paste its slug into `DTU_IMAGES_SLUG` / `DTU_POINTS_SLUG`. Kaggle's
   DTU mirrors come and go, and Kaggle will not serve a dataset's file list
   without an API token, so look at what one actually holds before trusting it:
   `jiezhu2/rectified` is 20.2 GB against the official 129.6 GB, which fits
   "all 124 scans at one lighting condition" exactly as well as it fits "~19
   scans at all seven". A mount that does not carry a scan, or does not carry
   `DTU_LIGHTING_GLOB` inside it, counts as absent for that scan and the
   archive fills the gap; it is never silently swapped for another scan.

   Section 8 pairs `Rectified/scanNN` with `stl0NN_total.ply` by scan number
   across every mount and everything fetched, so the two never need to live
   together.

   Datasets without a sensor ground truth — Tanks and Temples' *intermediate*
   set (M60 included, whose GT the benchmark withholds), NeRF-synthetic — leave
   B2/B4 emitting zero rows. They are not a substitute.
4. Leave `SCOPE = "smoke"` for the first run. Read section 10's table before
   anything else — it fails in seconds on data problems that otherwise surface
   three hours in.
5. **Save Version > Save & Run All (Commit)**. Results land in the Output tab as
   `tier_b_<run_id>.zip`.

**Resuming.** Re-running skips experiments whose CSV already exists, but a fresh
commit starts with an empty `/kaggle/working` — the skip only helps within a
session unless you attach the previous version's **output as an input** and point
`RESULTS_DIR` at it. The full lane is several sessions (see below), so set this
up before the first one.

### Colab

Runtime > Change runtime type > **T4** (free) or **L4/A100** (Pro). There is no
`/kaggle/input`, so either:

- set `FETCH_MISSING_DATASETS = True` and provide a Kaggle API token
  (`~/.kaggle/kaggle.json`, or `KAGGLE_USERNAME` / `KAGGLE_KEY` in the
  environment) — the notebook then pulls both slugs with `kagglehub`; or
- mount Drive, copy the data there, and point `DATA_ROOT` at that folder.

Mount Drive either way, or `OUT_DIR` dies with the runtime.

### What it costs

Estimates, not measurements — scaled from the one number this project has
measured (**62 s per MASt3R pair at 512 px** on its i7-1255U CPU) against
typical T4 throughput of roughly 0.5-0.8 s per pair. An L4 or A100 is about
2-3x faster than the figures below.

**Everything below the setup rows is per scene.** Every experiment loops the
scene list, and b4 multiplies its own grid (three frame-budget variants) by
it, so the lane is linear in `DTU_SCANS`. Section 12 prints the total for your
config, before anything runs — the numbers below are for orientation only.

| Stage | T4 | Notes |
|---|---|---|
| One-time setup (cells 2-7) | **12-18 min** | pip, MASt3R clone, RoPE CUDA build, 2.6 GB checkpoint |
| Fetching DTU by byte range (section 8) | **8-25 min** | ~1.2 GB for five scans: 0.57 GB of images and 0.61 GB of reference clouds, out of 136 GB of archives. Measured from Kaggle at 0.7 MB/s for the images and 1.9 MB/s for the clouds; the range block is 8 MB, which is ~4x fewer requests than the 1 MB that produced those numbers. Skipped for whatever a mount supplies, or `/kaggle/temp` already holds |
| Dataset test (sections 8-10) | **< 2 min** | plus the mount walk, seconds |
| **Smoke lane, b2 + b4, one scene** | **~50-80 min** | b2 is 4-6 min of it; b4 is 45-75 min on its own — three 40-frame reconstructions, and `DEFAULT_BUDGET` has no CLI override |
| **Full lane, b2 + b4, _per scene_** | see section 12 | linear in `DTU_SCANS`; section 12 computes the real total from `DTU_SCANS` / `FULL_SCENE_LIMIT` before anything runs |

Kaggle's GPU quota is ~30 h/week. With only two experiments in the lane, even
the full-lane grid comfortably fits inside a single session's 12 h ceiling for
the shipped three-scan config — trim `DTU_SCANS` or `FULL_SCENE_LIMIT["b4"]` if
you raise the scan count and section 12's estimate grows past that.

### What to look at first

`b2_reconstruction_accuracy.csv` has the headline numbers: Chamfer,
Hausdorff-95, precision/recall/F-score (at the manifest's `tau`), and normal
consistency, one row per scene. `b4_frame_budget_ablation.csv` is the more
interesting read — compare `V1_as_implemented` (motion-rank order into
pairing), `V2_motion_resort` (same selection, re-sorted into capture order)
and `V3_uniform` on the same metrics at matched frame count. If V2 beats V1 by
a real margin, that is a live ordering bug in `_cap_frames` feeding degraded
pairs into `swin` pairing, not a theoretical concern — worth a paragraph in
the paper either way.

## 1. Config

The only cell you should need to edit.

In [ ]:
# --- repo -----------------------------------------------------------------
REPO_URL = "https://github.com/AyushK0808/z-shift.git"
REPO_BRANCH = "main"

# --- data -----------------------------------------------------------------
# On Kaggle every attached dataset mounts read-only under /kaggle/input, so the
# root is the whole mount area. On Colab point this at a Drive folder, e.g.
# "/content/drive/MyDrive/zshift-tier-b".
DATA_ROOT = "/kaggle/input"

# Datasets to reconstruct, by Kaggle slug. Attach each one in the sidebar
# (Input > Add Input > paste the slug); it mounts at /kaggle/input/<name> and
# costs no working disk.
#
# `image_dir` is relative to the mount and is a *hint*. If it is missing or the
# mirror repackaged the tree, section 8 searches the mount for a folder named
# for the scene -- by name and by number, never "the biggest folder here", which
# on a mount holding ten DTU scans would hand one scan another scan's
# photographs. A scene it cannot identify is fetched, or dropped.
#
# `tau` is the F-score threshold in the dataset's own units, and every scene
# below must ship a sensor ground truth: B2/B3/B4/B6 skip a scene with
# gt_path=None outright and emit zero rows.
#
# DTU is what this protocol was written against -- structured-light ground
# truth, tau = 2 mm, and 124 distinct scenes rather than one. It ships images
# and ground truth as SEPARATE archives, so on Kaggle they arrive as two
# mounts; section 8 indexes GT across every mount and pairs Rectified/scanNN
# with Points/stl/stl0NN_total.ply by scan number.
#
# Both mounts are OPTIONAL. Whatever a mount does not supply is pulled from
# DTU's own server by byte range (DTU_IMAGES_URL / DTU_POINTS_URL below), so a
# slug here saves bandwidth rather than being the thing the lane depends on.
# A mount is used for a scan only when it holds that scan AND DTU_LIGHTING_GLOB
# matches inside it: a mirror carrying a different lighting condition, or a
# different subset of scans, is treated as absent rather than substituted for.
#
# Kaggle's DTU mirrors come and go, and Kaggle does not serve a dataset's file
# list without a token, so a slug is only worth setting once you have looked
# at it. jiezhu2/rectified, the obvious candidate, is 20.2 GB against the
# official archive's 129.6 GB -- definitely a subset, and 20.2 GB fits "all 124
# scans at one lighting condition" as readily as "~19 scans at all seven".
DTU_IMAGES_SLUG = ""  # e.g. "jiezhu2/rectified" -- check its scans on attach
DTU_POINTS_SLUG = ""  # a Points/stl mirror, if you have one; see DTU_POINTS_URL

# Ground truth without a mount. There is no dependable Kaggle mirror of DTU's
# STL reference clouds, and the official archive is 6.97 GB for 124 scans when
# we want ten. Apache serves byte ranges, so section 8 reads the archive's
# central directory and pulls only the members it needs -- 88-140 MB a scan,
# 0.61 GB for the five below, rather than 6.97 GB. Set to None to require an
# attached mount instead.
DTU_POINTS_URL = "https://roboimagedata2.compute.dtu.dk/data/MVS/Points.zip"

# Images without a mount, by the same route. Rectified.zip is 129.6 GB (124
# scans x 49 poses x 7 lighting conditions), the server serves byte ranges, and
# section 8 reads only what the scans below need: ~7 MB to read the archive's
# central directory, then 77-146 MB a scan for one lighting condition -- 0.57 GB
# for the five scans here. It lands in WORK_DIR, which is /kaggle/temp: roomy,
# ephemeral, and re-fetched each session like pip and the checkpoint.
# Set to None to require an attached mount instead.
DTU_IMAGES_URL = "https://roboimagedata2.compute.dtu.dk/data/MVS/Rectified.zip"

# Which scans to work on: ground truth is fetched for these, and each becomes a
# scene. The first ten of DTU's standard evaluation subset, so an F-score here
# is comparable with published DTU numbers.
#
# The full lane is linear in this tuple -- every experiment loops scenes, and
# b3/b4/b5/b7 multiply their own grid by it. FULL_SCENE_LIMIT below caps those,
# so the tuple sets the size of the b2/b6 aggregate and little else, and
# section 12 prints the hours it lands on.
#
# Three, from the head of the standard 15-scene DTU subset: three distinct
# objects is the minimum that gives a per-scene table plus a mean with a
# spread, and it matches FULL_SCENE_LIMIT below, which already caps every
# ablation at three -- so b2/b6/b8 no longer run a wider aggregate than the
# ablations do. Roughly 25-41 h against 29-49 h for five and 40-68 h for
# ten (cell 12 computes the real estimate from whatever is here). Extend it
# back towards (55, 63, 65, 69, 83, 97, 105) if the quota is there; the
# fetch follows whatever is here.
DTU_SCANS = (24, 37, 40)

# DTU photographs every viewpoint under 7 lighting conditions:
# rect_001_0_r5000.png .. rect_001_6_r5000.png. Taking the directory whole
# hands MASt3R seven near-duplicate copies of each pose, which multiplies the
# pair count and adds no parallax. Pick one condition. "*_3_r5000.png" is the
# usual choice (diffuse, evenly lit); "*_max.png" is the merged exposure if
# the mirror ships it.
DTU_LIGHTING_GLOB = "*_3_r5000.png"

# One scene per scan. Distinct objects, not one scene at several resolutions --
# which is what makes an aggregate across them mean something.
KAGGLE_SCENES = [
    {
        "slug": DTU_IMAGES_SLUG,
        "name": f"scan{scan}",
        # A hint. If a mirror repackaged the tree, section 8 looks for a folder
        # named for THIS scan; it never falls back to another scan's folder.
        "image_dir": f"Rectified/scan{scan}",
        # Left null on purpose: the GT lives on the other mount, and matching
        # by scan number across mounts is more robust than a guessed path.
        "gt_path": None,
        "image_glob": DTU_LIGHTING_GLOB,
        "tau": 2.0,
        "units": "mm",
        "notes": f"DTU scan{scan}, structured-light ground truth, tau = 2 mm (DTU's own)",
    }
    for scan in DTU_SCANS
]

# Scenes on a mount you point DATA_ROOT at directly, with paths relative to it.
# Left empty because KAGGLE_SCENES covers the attached datasets; fill it in for
# a DTU mount or your own capture:
#
# SCENES = [
#     {
#         "name": "scan24",
#         "image_dir": "dtu/Rectified/scan24",
#         "gt_path": "dtu/Points/stl/stl024_total.ply",
#         "tau": 2.0,
#         "units": "mm",
#         "notes": "DTU eval subset; sensor ground truth",
#     },
# ]
SCENES = []

# --- kaggle plumbing ------------------------------------------------------
# Pull a slug with kagglehub when it is not attached. Attaching is better: it
# mounts instantly, needs no internet and costs no working disk. On Colab there
# is no /kaggle/input at all, so this (plus a Kaggle API token) is the way in.
FETCH_MISSING_DATASETS = False

# If nothing above resolves, walk the mounts and build the scene list from what
# is actually there. Never silent: it logs what it picked, and the dataset test
# then runs against it.
AUTO_DISCOVER_SCENES = True

# Ceiling on auto-discovered scenes -- DTU ships 128 scans.
MAX_DISCOVERED_SCENES = 8

# tau and units for any scene that does not state its own. DTU's convention.
DEFAULT_TAU = 2.0
DEFAULT_UNITS = "mm"

# Stop the notebook when a dataset check FAILS. Warnings never stop it.
DATASET_TEST_STRICT = True

# --- scope ----------------------------------------------------------------
# "smoke" = minutes per experiment, validates the orchestration, NOT publishable.
# "full"  = the real grid.
SCOPE = "smoke"

# Trimmed to the two experiments the paper's results section actually needs:
# B2 (reconstruction accuracy baseline) and B4 (frame-budget ablation, the
# headline result -- it is the one that can surface a real ordering bug rather
# than just confirm an expected behaviour). The other six exp_b*.py modules
# still exist in bench/; add them back here (and to MODULES / SMOKE_ARGS /
# FULL_ARGS below, and to section 14's run cells) if you need them again.
EXPERIMENTS = ["b2", "b4"]

# Scenes per experiment in the full lane. b4 multiplies its own grid (three
# frame-budget variants) by every scene, which is where a ten-scan tuple goes
# from long to a lot of Kaggle quota. Three scans show the trend. b2 wants
# every scene -- it is the accuracy aggregate -- so it is absent from this
# dict; an experiment absent from this dict runs on every scene.
FULL_SCENE_LIMIT = {"b4": 3}

# Per-experiment ceiling. A run that blows through this is a bug, not progress.
#
# The smoke lane is one scene, so smoke numbers are absolute minutes. The full
# lane runs each experiment once per scene, so ITS numbers are PER SCENE and
# section 12 multiplies them by the scenes that experiment actually gets: a
# flat ceiling starts killing correct runs the moment DTU_SCANS grows.
TIMEOUT_MIN = {
    "smoke": {"b2": 30, "b4": 120},
    "full": {"b2": 120, "b4": 480},
}

# Stop LAUNCHING new experiments once the session is this old (minutes).
# Kaggle kills at 12 h (720 min); leave room to collect results.
SESSION_BUDGET_MIN = 660

# Delete each experiment's reconstruction outputs after it finishes. The sparse
# alignment cache is ~1.5 GB per run and accumulates fast.
CLEAN_WORK_AFTER_EACH = True

# Force numpy < 2 to match pyproject. Needs a kernel restart and can break
# preinstalled wheels compiled against numpy 2 -- only set this if the import
# check in the preflight cell actually fails.
PIN_NUMPY_LT2 = False

# Re-run experiments whose CSV already exists.
FORCE_RERUN = False

## 2. Paths and logging

Sets up the log tree and installs handlers that catch tracebacks from notebook
cells, from `warnings`, and from native crashes.

In [ ]:
import datetime as _dt
import faulthandler
import json
import logging
import os
import platform
import shutil
import subprocess
import sys
import time
import traceback
from collections import deque
from pathlib import Path

faulthandler.enable()


def _detect_platform():
    if Path("/kaggle/working").exists():
        return "kaggle"
    if "google.colab" in sys.modules or Path("/content").exists():
        return "colab"
    return "local"


PLATFORM = _detect_platform()

if PLATFORM == "kaggle":
    # /kaggle/working is persisted but capped (~20 GB) -- results only.
    # /kaggle/temp is ephemeral and roomy -- caches and reconstructions.
    OUT_DIR = Path("/kaggle/working/tier_b")
    WORK_DIR = Path("/kaggle/temp/zshift")
    REPO_DIR = Path("/kaggle/working/z-shift")
elif PLATFORM == "colab":
    drive = Path("/content/drive/MyDrive")
    OUT_DIR = (drive / "zshift-tier-b-results") if drive.exists() else Path("/content/tier_b")
    WORK_DIR = Path("/content/zshift-work")
    REPO_DIR = Path("/content/z-shift")
else:
    OUT_DIR = Path.cwd() / "tier_b_out"
    WORK_DIR = Path.cwd() / "tier_b_work"
    REPO_DIR = Path.cwd()

RESULTS_DIR = OUT_DIR / "results"
RUN_ID = _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
LOG_DIR = OUT_DIR / "logs" / RUN_ID
HF_HOME = WORK_DIR / "hf"

for d in (OUT_DIR, WORK_DIR, RESULTS_DIR, LOG_DIR, HF_HOME):
    d.mkdir(parents=True, exist_ok=True)

SESSION_LOG = LOG_DIR / "session.log"
CRASH_LOG = LOG_DIR / "faulthandler.log"

# Held open for the whole session so a native crash has somewhere to dump.
_crash_fh = open(CRASH_LOG, "w")  # noqa: SIM115
faulthandler.enable(file=_crash_fh, all_threads=True)

log = logging.getLogger("tierb")
log.setLevel(logging.DEBUG)
log.handlers.clear()
log.propagate = False

_fmt = logging.Formatter("%(asctime)s %(levelname)-8s %(name)s: %(message)s", datefmt="%H:%M:%S")
_fh = logging.FileHandler(SESSION_LOG, encoding="utf-8")
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
_sh = logging.StreamHandler(sys.stdout)
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
log.addHandler(_fh)
log.addHandler(_sh)

logging.captureWarnings(True)
_warn_log = logging.getLogger("py.warnings")
_warn_log.handlers.clear()
_warn_log.addHandler(_fh)


def _log_uncaught(exc_type, exc, tb):
    log.error("uncaught exception", exc_info=(exc_type, exc, tb))


sys.excepthook = _log_uncaught

# IPython swallows sys.excepthook, so hook its traceback printer too. Without
# this, a cell that raises leaves nothing in session.log.
try:
    _ip = get_ipython()  # noqa: F821
except NameError:
    _ip = None
if _ip is not None and not getattr(_ip, "_tierb_hooked", False):
    _orig_showtraceback = _ip.showtraceback

    def _showtraceback(*args, **kwargs):
        log.error("cell raised", exc_info=sys.exc_info())
        return _orig_showtraceback(*args, **kwargs)

    _ip.showtraceback = _showtraceback
    _ip._tierb_hooked = True

log.info("platform=%s run_id=%s", PLATFORM, RUN_ID)
log.info("repo=%s", REPO_DIR)
log.info("work=%s (ephemeral: caches, reconstructions)", WORK_DIR)
log.info("out=%s (persisted: CSVs, logs)", OUT_DIR)
log.info("session log -> %s", SESSION_LOG)

## 3. Command runner

Streams output live, tees every line to a log file, caps how much reaches the
notebook (a 3 h run must not bloat the `.ipynb`), and turns a non-zero exit into
an exception carrying the tail of the output.

In [ ]:
import threading

MAX_ECHO_LINES = 400  # per command; the log file always gets everything


def describe_exit(rc):
    if rc == 0:
        return "ok"
    if rc == -9:
        return "SIGKILL -- almost always the OOM killer (host RAM, not VRAM)"
    if rc == -11:
        return "SIGSEGV -- native crash; check faulthandler.log for Python frames"
    if rc == -6:
        return "SIGABRT -- native abort (CUDA / VTK)"
    if rc < 0:
        return "killed by signal " + str(-rc)
    return "non-zero exit " + str(rc)


class CommandFailed(RuntimeError):
    def __init__(self, cmd, returncode, tail, log_path, timed_out=False, timeout_s=None):
        self.cmd = cmd
        self.returncode = returncode
        self.tail = tail
        self.log_path = log_path
        self.timed_out = timed_out
        self.timeout_s = timeout_s
        # A timeout kill also lands as SIGKILL on Linux. Say which one it was,
        # or every timeout gets misread as the OOM killer.
        if timed_out:
            self.reason = "timed out after %.1f min and was killed" % ((timeout_s or 0) / 60)
        else:
            self.reason = describe_exit(returncode)
        super().__init__(f"{self.reason}\n{tail}")


def sh(
    cmd, *, cwd=None, log_path=None, env=None, timeout_s=None, check=True, echo=True, tail_lines=60
):
    # Run cmd (list, or str via bash -lc), tee output to log_path,
    # return (returncode, tail_text).
    if isinstance(cmd, str):
        cmd = ["bash", "-lc", cmd]
    log_path = Path(log_path) if log_path else (LOG_DIR / "commands.log")
    log_path.parent.mkdir(parents=True, exist_ok=True)

    full_env = dict(os.environ)
    if env:
        full_env.update({k: str(v) for k, v in env.items()})

    printable = " ".join(str(c) for c in cmd)
    header = (
        "\n"
        + "=" * 78
        + "\n$ "
        + printable
        + "\n  cwd="
        + str(cwd)
        + "  started="
        + _dt.datetime.now().isoformat(timespec="seconds")
        + "\n"
        + "=" * 78
        + "\n"
    )
    log.debug("running: %s", printable)

    tail = deque(maxlen=max(tail_lines, 200))
    echoed = [0]
    start = time.monotonic()

    with open(log_path, "a", encoding="utf-8", errors="replace") as fh:
        fh.write(header)
        fh.flush()
        proc = subprocess.Popen(  # noqa: S603
            [str(c) for c in cmd],
            cwd=str(cwd) if cwd else None,
            env=full_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            errors="replace",
        )

        def pump():
            try:
                for line in proc.stdout:
                    fh.write(line)
                    tail.append(line)
                    if echo:
                        if echoed[0] < MAX_ECHO_LINES:
                            print(line, end="")
                            echoed[0] += 1
                        elif echoed[0] == MAX_ECHO_LINES:
                            print("... output continues in " + str(log_path))
                            echoed[0] += 1
                fh.flush()
            except ValueError:
                pass  # fh already closed: main thread gave up waiting on a stuck grandchild

        pumper = threading.Thread(target=pump, daemon=True)
        pumper.start()

        timed_out = False
        try:
            rc = proc.wait(timeout=timeout_s)
        except subprocess.TimeoutExpired:
            timed_out = True
            log.error("TIMEOUT after %.1f min, killing: %s", (timeout_s or 0) / 60, cmd[0])
            proc.kill()
            rc = proc.wait()
        pumper.join(timeout=30)
        if pumper.is_alive():
            log.warning(
                "pump thread still draining after 30s for %s (grandchild may hold stdout open)",
                printable,
            )
        elapsed = time.monotonic() - start
        fh.write(f"\n--- exit {rc} ({describe_exit(rc)}) after {elapsed:.1f}s ---\n")

    tail_text = "".join(list(tail)[-tail_lines:])
    if timed_out or (check and rc != 0):
        raise CommandFailed(cmd, rc, tail_text, log_path, timed_out=timed_out, timeout_s=timeout_s)
    return rc, tail_text

## 4. Machine report

Recorded so a number in a CSV can never be separated from the box that produced
it. `env_metadata()` already stamps `gpu`, `torch` and `git_commit` onto every
results row; this is the same information, up front, in the log.

In [ ]:
def report_machine():
    log.info("python  %s", platform.python_version())
    log.info("os      %s", platform.platform())
    try:
        import torch

        log.info("torch   %s (cuda %s)", torch.__version__, torch.version.cuda)
        log.info("cuda available: %s", torch.cuda.is_available())
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                props = torch.cuda.get_device_properties(i)
                log.info(
                    "gpu %d   %s, %.1f GB, cc %d.%d",
                    i,
                    props.name,
                    props.total_memory / 1024**3,
                    props.major,
                    props.minor,
                )
    except Exception:
        log.exception("torch import failed")
    try:
        import numpy

        log.info("numpy   %s", numpy.__version__)
    except Exception:
        log.exception("numpy import failed")

    for label, path in (("work", WORK_DIR), ("out", OUT_DIR)):
        usage = shutil.disk_usage(path)
        log.info(
            "disk %-5s %.1f GB free of %.1f GB  (%s)",
            label,
            usage.free / 1024**3,
            usage.total / 1024**3,
            path,
        )
    try:
        rc, _ = sh("nvidia-smi", log_path=LOG_DIR / "setup.log", check=False)
        if rc != 0:
            log.warning("nvidia-smi returned %s -- is a GPU accelerator selected?", rc)
    except FileNotFoundError:
        log.error("nvidia-smi not found: this runtime has no GPU. Stop and enable one.")


report_machine()

## 5. Clone the repo

No `pip install -e .` of the project. Two reasons:

- `pyproject.toml` pins `requires-python >=3.11,<3.12`, which fails outright on
  a 3.12 image;
- its `torch` / `torchvision` entries resolve from PyPI on Linux (the CUDA index
  is scoped to `sys_platform == 'win32'`), which would replace the preinstalled
  CUDA build with a CPU wheel and quietly cost you the GPU.

Setting `PYTHONPATH` to the repo root plus `src/` imports both `bench` and
`spatial_ingestion` with none of that risk. Nothing in Tier B uses the
`zshift-*` console scripts.

In [ ]:
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    log.info("repo already present, fetching")
    sh(["git", "fetch", "--all", "--prune"], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log")
    sh(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log")
    sh(["git", "pull", "--ff-only"], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log", check=False)
else:
    sh(
        ["git", "clone", "--branch", REPO_BRANCH, "--depth", "50", REPO_URL, str(REPO_DIR)],
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )

_, _commit = sh(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, log_path=LOG_DIR / "setup.log", echo=False
)
COMMIT = _commit.strip().splitlines()[-1] if _commit.strip() else "unknown"
log.info("repo at commit %s", COMMIT)

MAST3R_DIR = REPO_DIR / "third_party" / "mast3r"
DUST3R_DIR = MAST3R_DIR / "dust3r"

# Repo root for `bench`, src/ for `spatial_ingestion`.
PYTHONPATH = os.pathsep.join([str(REPO_DIR), str(REPO_DIR / "src")])

CHILD_ENV = {
    "PYTHONPATH": PYTHONPATH,
    "PYTHONUNBUFFERED": "1",
    "PYTHONFAULTHANDLER": "1",
    "HF_HOME": str(HF_HOME),
    "HUGGINGFACE_HUB_CACHE": str(HF_HOME / "hub"),
    "TOKENIZERS_PARALLELISM": "false",
    # PyVista/VTK are used for mesh filtering, never rendering, but this
    # runtime is headless -- make the intent explicit.
    "PYVISTA_OFF_SCREEN": "true",
    "MPLBACKEND": "Agg",
}

missing = [p for p in ("bench", "src/spatial_ingestion") if not (REPO_DIR / p).exists()]
if missing:
    raise SystemExit(
        f"missing from the clone: {missing}. These are untracked on the laptop -- "
        "commit and push them, then re-run."
    )
log.info("bench/ and src/spatial_ingestion/ present")

## 6. Dependencies

Read straight out of `pyproject.toml`, minus `torch`/`torchvision` (keep the
preinstalled CUDA build) and minus `numpy` unless you opted into the `<2` pin.

In [ ]:
try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib  # type: ignore

with open(REPO_DIR / "pyproject.toml", "rb") as fh:
    _pyproject = tomllib.load(fh)

SKIP_PREFIXES = ("torch", "torchvision")
deps = []
for dep in _pyproject["project"]["dependencies"]:
    name = dep.split(">")[0].split("<")[0].split("=")[0].split("[")[0].strip().lower()
    if name.startswith(SKIP_PREFIXES):
        log.info("skipping %-12s (keeping the preinstalled CUDA build)", name)
        continue
    if name == "numpy" and not PIN_NUMPY_LT2:
        log.info("skipping %-12s (set PIN_NUMPY_LT2 only if imports actually fail)", name)
        continue
    deps.append(dep)

log.info("installing %d dependencies", len(deps))
sh(
    [sys.executable, "-m", "pip", "install", "-q", *deps],
    log_path=LOG_DIR / "setup.log",
    timeout_s=1800,
)
log.info("dependencies installed")
if PIN_NUMPY_LT2:
    log.warning("numpy pinned <2 -- RESTART THE KERNEL now, then re-run from cell 2")

## 7. MASt3R

Mirrors `scripts/setup-mast3r.sh` (same pinned commit) but installs with
`--no-deps`: MASt3R's and DUSt3R's own `requirements.txt` list `torch`, and
letting pip resolve them is the other way to lose the CUDA build. Their real
dependencies are already in the project's list, which is why `roma`, `einops`,
`pyglet<2` and friends are there.

The RoPE CUDA kernel compile fails on your laptop and should succeed here. That
is a genuinely different code path from the one Tier A measured, and belongs in
the paper's setup section.

In [ ]:
PINNED_MAST3R = "f5209afc300cec36239a7ac992263f36847bbba0"

PY_STUB = """[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "{name}"
version = "0.1.0"
requires-python = ">=3.10"

[tool.setuptools.packages.find]
where = ["."]
include = ["{name}*"]
"""

if not MAST3R_DIR.exists():
    sh(
        ["git", "clone", "https://github.com/naver/mast3r", str(MAST3R_DIR)],
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )
    sh(["git", "checkout", PINNED_MAST3R], cwd=MAST3R_DIR, log_path=LOG_DIR / "setup.log")
    sh(
        ["git", "submodule", "update", "--init", "--recursive"],
        cwd=MAST3R_DIR,
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )
else:
    log.info("mast3r already cloned at %s", MAST3R_DIR)

for target, name in ((MAST3R_DIR, "mast3r"), (DUST3R_DIR, "dust3r")):
    stub = target / "pyproject.toml"
    if not stub.exists():
        stub.write_text(PY_STUB.format(name=name), encoding="utf-8")
        log.info("wrote %s", stub)
    sh(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(target)],
        log_path=LOG_DIR / "setup.log",
        timeout_s=900,
    )

# RoPE CUDA kernels: a speedup, not a requirement. Never fatal.
curope = DUST3R_DIR / "croco" / "models" / "curope"
try:
    sh(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=curope,
        log_path=LOG_DIR / "setup.log",
        timeout_s=1800,
    )
    log.info("RoPE CUDA kernels compiled")
    CUROPE_BUILT = True
except CommandFailed as exc:
    CUROPE_BUILT = False
    log.warning(
        "RoPE kernel compile failed (%s); falling back to the PyTorch path. "
        "Not fatal, but note it in the setup section. Log: %s",
        describe_exit(exc.returncode),
        exc.log_path,
    )

## 8. Kaggle datasets

Kaggle mounts every attached dataset read-only at `/kaggle/input/<slug>`, one
directory per dataset, with no re-download between sessions. This cell decides
which scenes Tier B runs on:

1. resolve `SCENES` against `DATA_ROOT`, and every `KAGGLE_SCENES` entry
   against the mount its slug names;
2. fetch the images for whatever did not resolve, by byte range out of
   `DTU_IMAGES_URL` — 77–146 MB a scan for one lighting condition, out of a
   129.6 GB archive;
3. fetch the reference clouds for the scans that got that far, by byte range
   out of `DTU_POINTS_URL` — 88–140 MB a scan out of 6.97 GB. Images first, so
   a scan that is about to be dropped never costs a cloud download;
4. pair each scene with a cloud by scan number, across every mount and
   everything fetched;
5. if *nothing* resolved and `AUTO_DISCOVER_SCENES` is on, walk `DATA_ROOT` and
   then every mount, and build the scene list from what is actually there.

**A named scene is never substituted.** The `image_dir` hint is a hint, and a
mirror that repackaged the tree is fine — but the search that follows matches
the scene's own name and number (`scan37` = `scan037` = `Scan37`, never
`scan3`), and requires `image_glob` to match inside it. Ten DTU scans share one
mount, so "the biggest image folder here" would hand one scan another scan's
photographs, keep its name, and pair it with its own reference cloud: an
F-score over two different objects, in a row that looks exactly like a real
one. A scan that cannot be identified is fetched, and failing that, dropped
with a warning — section 10's `requested_scans` check says which.

**No dataset slug is hardcoded.** Redistributions of DTU and BlendedMVS appear
and disappear from Kaggle, so the notebook reads the mount rather than a list of
names. Attach whatever you have; these layouts are recognised:

| Layout | Looks like | Scene name | Ground truth matched by |
|---|---|---|---|
| DTU | `Rectified/scan24/*.png` + `Points/stl/stl024_total.ply` | `scan24` | scan number, anywhere on the mount |
| BlendedMVS | `<id>/blended_images/*.jpg` | `<id>` | a cloud beside the images |
| Tanks and Temples, COLMAP | `<scene>/images/*.jpg` + `<scene>/*.ply` | `<scene>` | one level up from the images |
| ETH3D | `<scene>/images/dslr_images/*.JPG` + laser-scan `<scene>/*.ply` | `<scene>` | one level up from the images -- real DSLR/phone EXIF, unlike DTU and T&T |
| own capture | `captures/desk_orbit/frames/*.jpg` | `desk_orbit` | a cloud beside the images |
| anything else | a folder holding >= 4 images | the folder | beside, or one level up |

`FETCH_MISSING_DATASETS` pulls a slug with `kagglehub` for a dataset you did not
attach. That needs *Settings > Internet* on and writes to the working disk, so
attaching is still the better option — and for DTU specifically, the byte-range
fetch above is cheaper than either, because it reads 1.2 GB out of the archive
instead of mirroring all of it.

A mount with no `.ply` gives you **no ground truth**, and B2/B4 skip
scenes with `gt_path=None` — they exit 0 having written zero rows. This cell
says so loudly rather than letting you find out three hours in.

In [ ]:
import fnmatch
import io
import re
import urllib.request
import zipfile

# Kaggle mounts every attached dataset read-only at /kaggle/input/<slug>, one
# directory per dataset. Discovery reads the mount instead of a hardcoded list
# of dataset names: redistributions of DTU and BlendedMVS appear and disappear
# from Kaggle, so a manifest pinned to one particular slug rots.

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
GT_SUFFIXES = {".ply", ".obj", ".off", ".stl", ".xyz", ".pcd"}

# A suffix filter cannot tell a photograph from a depth map -- .png is .png.
# NeRF-synthetic ships r_0_depth_0000.png and r_0_normal_0000.png beside the
# RGB frames, and feeding those to MASt3R as photographs corrupts the
# reconstruction without failing anything. Exclude them by name and by folder.
NON_PHOTO_RE = re.compile(
    r"(?:^|[_-])(depth|normals?|masks?|alpha|segs?|segmentation)(?:[_-]|\d|$)",
    re.IGNORECASE,
)
NON_PHOTO_DIRS = {
    "depth",
    "depths",
    "normal",
    "normals",
    "mask",
    "masks",
    "alpha",
    "seg",
    "segmentation",
}


def _is_photo(path):
    path = Path(path)
    if path.suffix.lower() not in IMAGE_SUFFIXES:
        return False
    if NON_PHOTO_RE.search(path.stem):
        return False
    return path.parent.name.lower() not in NON_PHOTO_DIRS


# A directory holding at least this many images is a candidate scene.
MIN_SCENE_IMAGES = 4

# Directory names that mean "the images live here" -- such a scene is named
# after its parent instead. Covers DTU (Rectified/scan24/), BlendedMVS
# (<id>/blended_images/), COLMAP and Tanks and Temples (<scene>/images/),
# ETH3D (<scene>/images/dslr_images/, real EXIF unlike DTU/T&T) and our own
# extracted captures (<scene>/frames/).
IMAGE_DIR_NAMES = {
    "images",
    "image",
    "rect",
    "rectified",
    "blended_images",
    "frames",
    "rgb",
    "color",
    "dense",
    "dslr_images",
    "train",
    "test",
    "val",
}

# COLMAP ships images/ beside downsampled images_2/, images_4/, images_8/ --
# the same scene at four resolutions, sharing one ground truth. Discovery would
# otherwise report four independent scenes, so B2/B3/B4/B6 emit four rows that
# read as replication and are one scene counted four times.
#
# MASt3R resizes the long side to --image-size (512) regardless, so a quarter
# resolution level costs nothing in quality and decodes far faster than
# 4096x2160. Set to "images" to keep full resolution instead.
PYRAMID_RE = re.compile(r"^(?:images?|rgb|color)(?:_\d+)?$", re.IGNORECASE)
PREFERRED_PYRAMID = "images_4"

# Walking a 50 GB mount must not become the slow part of the notebook.
MAX_WALK_DIRS = 20000
MAX_GT_FILES = 5000

KAGGLE_INPUT = Path("/kaggle/input")


def _images_in(directory, glob=None):
    # `glob` mirrors Scene.image_glob: DTU needs one lighting condition out of
    # seven, and the count reported here has to be the count the run uses.
    directory = Path(directory)
    try:
        candidates = directory.glob(glob) if glob else directory.iterdir()
        return sorted(p for p in candidates if _is_photo(p))
    except OSError:
        return []


def _n_dropped(directory, glob=None):
    # How many image-suffixed files _is_photo rejected, reported per scene
    # below so the exclusion is never silent.
    directory = Path(directory)
    try:
        candidates = directory.glob(glob) if glob else directory.iterdir()
        suffixed = sum(1 for p in candidates if p.suffix.lower() in IMAGE_SUFFIXES)
    except OSError:
        return 0
    return suffixed - len(_images_in(directory, glob))


def _first_number(text):
    match = re.search(r"(\d+)", str(text))
    return int(match.group(1)) if match else None


def scan_tree(root):
    # One walk per mount: every candidate image dir, and every GT-shaped file.
    image_dirs, gt_files, truncated = [], [], False
    for n_dirs, (dirpath, dirnames, filenames) in enumerate(os.walk(root), start=1):
        dirnames[:] = sorted(d for d in dirnames if not d.startswith("."))
        if n_dirs > MAX_WALK_DIRS:
            truncated = True
            break
        here = Path(dirpath)
        n_images = sum(1 for f in filenames if _is_photo(here / f))
        if n_images >= MIN_SCENE_IMAGES:
            image_dirs.append((here, n_images))
        if len(gt_files) < MAX_GT_FILES:
            gt_files.extend(here / f for f in filenames if Path(f).suffix.lower() in GT_SUFFIXES)
    return image_dirs, gt_files, truncated


_TREE_CACHE = {}


def cached_tree(root):
    # Every walk here crosses a mount that can be tens of GB, and resolution
    # now asks for one per scene rather than one per notebook. Invalidated by
    # hand whenever this notebook writes into a root.
    key = str(root)
    if key not in _TREE_CACHE:
        _TREE_CACHE[key] = scan_tree(root)
    return _TREE_CACHE[key]


def _invalidate_tree(root):
    _TREE_CACHE.pop(str(root), None)


_GT_INDEX = None


def gt_files_everywhere():
    # DTU ships Rectified (images) and Points (ground truth) as separate
    # archives, so on Kaggle they are two mounts. A GT search confined to the
    # image mount finds nothing, gt_path stays None, and B2/B3/B4/B6 skip every
    # scene while reporting a clean exit. Index once, across everything.
    global _GT_INDEX
    if _GT_INDEX is None:
        seen, files = set(), []
        for root in (Path(DATA_ROOT), *FETCHED_ROOTS, *MOUNTS):
            root = Path(root)
            if root in seen or not root.is_dir():
                continue
            seen.add(root)
            files.extend(cached_tree(root)[1])
        _GT_INDEX = files
        log.info(
            "ground-truth index: %d candidate file(s) across %d root(s)", len(files), len(seen)
        )
    return _GT_INDEX


def _match_gt(scene_name, image_dir, gt_files):
    # Beside the images first, then one level up (COLMAP, Tanks and Temples),
    # then by scan number anywhere on the mount -- which is what DTU needs:
    # Rectified/scan24/ and Points/stl/stl024_total.ply share nothing but "24".
    beside = [g for g in gt_files if g.parent in (image_dir, image_dir.parent)]
    if beside:
        return max(beside, key=lambda p: p.stat().st_size)
    number = _first_number(scene_name)
    if number is None:
        return None
    numbered = [g for g in gt_files if _first_number(g.stem) == number]
    return max(numbered, key=lambda p: p.stat().st_size) if numbered else None


def _drop_pyramid_duplicates(image_dirs):
    # Siblings whose names differ only by COLMAP's _N suffix AND that hold the
    # same number of images are one scene at several resolutions. Differing
    # counts mean genuinely different scenes, so grouping on the count keeps
    # this conservative.
    groups, passthrough = {}, []
    for item in image_dirs:
        directory, n_images = item
        if PYRAMID_RE.match(directory.name):
            groups.setdefault((directory.parent, n_images), []).append(item)
        else:
            passthrough.append(item)

    kept = []
    for (parent, _n), members in groups.items():
        if len(members) == 1:
            kept.append(members[0])
            continue
        by_name = {directory.name.lower(): item for item, directory in ((m, m[0]) for m in members)}
        choice = by_name.get(PREFERRED_PYRAMID.lower()) or by_name.get("images")
        if choice is None:
            choice = sorted(members, key=lambda item: str(item[0]))[0]
        log.warning(
            "%s: %d resolution levels of one scene (%s) -- keeping %s. They share "
            "a ground truth, so counting them separately would report one scene "
            "%d times. Set PREFERRED_PYRAMID to choose a different level.",
            parent.name,
            len(members),
            ", ".join(sorted(by_name)),
            choice[0].name,
            len(members),
        )
        kept.append(choice)
    return passthrough + kept


def _scene_dir_matches(directory, name):
    # scan37 == scan037 == Scan37, but never scan3 and never stl037: the number
    # has to match and the non-numeric part has to be the same word.
    label = directory.name.lower()
    wanted = name.lower()
    if label == wanted:
        return True
    number = _first_number(wanted)
    if number is None or _first_number(label) != number:
        return False
    return re.sub(r"\d+", "", label) == re.sub(r"\d+", "", wanted)


def _find_scene_dir(root, name, glob):
    """Locate the directory for a *named* scene under root, or None.

    Deliberately never falls back to the largest image folder around. Ten DTU
    scans share one mount, so a scan whose folder is absent would take another
    scan's photographs, keep its own name, and be paired with its own reference
    cloud by _match_gt -- an F-score over two different objects, in a row that
    looks exactly like a real one.
    """
    named = [item for item in cached_tree(root)[0] if _scene_dir_matches(item[0], name)]
    if not named:
        return None
    matched = [item for item in named if len(_images_in(item[0], glob)) >= MIN_SCENE_IMAGES]
    if not matched:
        # The folder is here and the lighting condition is not: a mirror that
        # kept *_max.png answers this way for every scan it carries.
        log.warning(
            "%s: %s exists under %s but %s matches %d file(s) there -- treating the "
            "scene as absent rather than reconstructing something else",
            name,
            named[0][0].name,
            root,
            glob or "*",
            len(_images_in(named[0][0], glob)),
        )
        return None
    return max(matched, key=lambda item: (item[1], str(item[0])))[0]


def discover_scenes(root, limit=None, used_names=None):
    limit = MAX_DISCOVERED_SCENES if limit is None else limit
    used_names = set() if used_names is None else used_names
    root = Path(root)
    if not root.is_dir() or limit <= 0:
        return []
    image_dirs, gt_files, truncated = cached_tree(root)
    image_dirs = _drop_pyramid_duplicates(image_dirs)
    known = set(gt_files)
    gt_files = gt_files + [g for g in gt_files_everywhere() if g not in known]
    if truncated:
        log.warning(
            "stopped walking %s after %d directories -- discovery is partial; "
            "point DATA_ROOT at the subtree you care about",
            root,
            MAX_WALK_DIRS,
        )

    # Match ground truth before ranking, then put the scenes that have it
    # first. B2/B4 skip a scene with gt_path=None outright, so letting a
    # GT-less scene take a MAX_DISCOVERED_SCENES slot ahead of one with GT
    # costs both experiments a row and buys nothing.
    candidates = []
    for image_dir, n_images in image_dirs:
        label = image_dir.name
        if (
            label.lower() in IMAGE_DIR_NAMES or PYRAMID_RE.match(label)
        ) and image_dir.parent != root:
            label = image_dir.parent.name
        candidates.append((image_dir, n_images, label, _match_gt(label, image_dir, gt_files)))
    candidates.sort(key=lambda item: (item[3] is None, -item[1], str(item[0])))

    found = []
    for image_dir, n_images, label, gt in candidates:
        name, suffix = label, 2
        while name in used_names:
            name = f"{label}_{suffix}"
            suffix += 1
        used_names.add(name)
        found.append(
            {
                "name": name,
                "image_dir": str(image_dir),
                "gt_path": str(gt) if gt else None,
                "tau": DEFAULT_TAU,
                "units": "unknown (auto-discovered)",
                # Discovery found this scene on a mount; nothing told it what a
                # coordinate means here. DEFAULT_TAU is DTU's 2 mm, which is a
                # placeholder on any other dataset -- section 10 warns rather
                # than passing this silently.
                "tau_is_default": True,
                "notes": f"auto-discovered under {root}; tau and units unverified",
                "n_images": n_images,
            }
        )
        if len(found) >= limit:
            break
    return found


def resolve_configured(scenes, data_root):
    # SCENES holds paths relative to DATA_ROOT. Resolve them to absolute paths
    # and say which ones are absent here, rather than failing later.
    data_root = Path(data_root)
    resolved, missing = [], []
    for scene in scenes:
        image_dir = (data_root / scene["image_dir"]).resolve()
        if not image_dir.is_dir():
            missing.append(f"{scene['name']}: {image_dir}")
            continue
        gt = scene.get("gt_path")
        gt_path = (data_root / gt).resolve() if gt else None
        if gt_path is not None and not gt_path.exists():
            log.warning("%s: gt_path does not exist: %s", scene["name"], gt_path)
            gt_path = None
        entry = dict(scene)
        entry["image_dir"] = str(image_dir)
        entry["gt_path"] = str(gt_path) if gt_path else None
        entry["n_images"] = len(_images_in(image_dir, scene.get("image_glob")))
        resolved.append(entry)
    return resolved, missing


FETCHED_ROOTS = []


def fetch_kaggle_datasets(slugs):
    # Optional. Attaching a dataset in the sidebar is better: it mounts
    # instantly and needs no internet. This is for a slug you did not attach,
    # and for Colab, which has no /kaggle/input at all.
    downloaded = []
    for slug in slugs:
        try:
            import kagglehub
        except ModuleNotFoundError:
            sh(
                [sys.executable, "-m", "pip", "install", "-q", "kagglehub"],
                log_path=LOG_DIR / "setup.log",
                timeout_s=600,
            )
            import kagglehub
        try:
            path = Path(kagglehub.dataset_download(slug))
        except Exception:
            log.exception(
                "could not download %s -- attach it as a dataset instead, "
                "or turn on Settings > Internet",
                slug,
            )
            continue
        log.info("downloaded %-40s -> %s", slug, path)
        downloaded.append(path)
        FETCHED_ROOTS.append(path)
    return downloaded


def _mount_for(slug):
    # Kaggle mounts a dataset under the second half of its slug.
    name = slug.split("/")[-1]
    direct = KAGGLE_INPUT / name
    if direct.is_dir():
        return direct
    # A renamed or hand-uploaded copy: match on the slug's distinctive words.
    words = [word for word in re.split(r"[-_]", name) if len(word) > 3][:2]
    for mount in MOUNTS:
        if words and all(word in mount.name.lower() for word in words):
            return mount
    return None


def resolve_entry(entry, root, source=None, allow_any_folder=False):
    # One configured entry against one root -- a mount, or a directory this
    # notebook fetched into. None means this root cannot supply it, and the
    # caller tries the next root or drops the scene.
    root = Path(root)
    glob = entry.get("image_glob")
    hinted = root / (entry.get("image_dir") or "")
    image_dir = hinted if _images_in(hinted, glob) else _find_scene_dir(root, entry["name"], glob)
    if image_dir is None and allow_any_folder:
        log.warning("%s: no images at %s -- walking %s instead", entry["name"], hinted, root)
        found = discover_scenes(root, limit=1)
        if found:
            image_dir = Path(found[0]["image_dir"])
    if image_dir is None:
        return None

    gt_hint = entry.get("gt_path")
    gt_path = root / gt_hint if gt_hint else None
    if gt_path is not None and not gt_path.exists():
        log.warning("%s: gt_path does not exist: %s", entry["name"], gt_path)
        gt_path = None

    # Ground truth is attached later, once the reference clouds have been
    # fetched for the scans that got this far.
    return {
        "name": entry["name"],
        "image_dir": str(image_dir),
        "gt_path": str(gt_path) if gt_path else None,
        "image_glob": glob,
        "tau": entry.get("tau", DEFAULT_TAU),
        "units": entry.get("units", DEFAULT_UNITS),
        "notes": f"{entry.get('notes', '')} [{source or root}]".strip(),
        "n_images": len(_images_in(image_dir, glob)),
    }


def resolve_kaggle_scenes(entries):
    # Resolve each configured entry against the mount its slug names. Anything
    # that does not resolve comes back as an entry rather than a string: the
    # image fetch below takes another run at it before it is dropped.
    per_slug = {}
    for entry in entries:
        per_slug[entry["slug"]] = per_slug.get(entry["slug"], 0) + 1

    resolved, unresolved = [], []
    for entry in entries:
        slug = entry["slug"]
        mount = _mount_for(slug) if slug else None
        if mount is None and slug and FETCH_MISSING_DATASETS:
            fetched = fetch_kaggle_datasets([slug])
            mount = fetched[0] if fetched else None
        if mount is None:
            reason = f"{slug} is not attached" if slug else "no image slug set"
            unresolved.append(dict(entry, reason=reason))
            continue
        scene = resolve_entry(
            entry,
            mount,
            source=f"kaggle: {slug}",
            # "whatever images are on this mount" is only an unambiguous answer
            # when the mount carries one configured scene and it is unnumbered.
            # Ten numbered DTU scans on one mount is the case it gets wrong.
            allow_any_folder=per_slug[slug] == 1 and _first_number(entry["name"]) is None,
        )
        if scene is None:
            unresolved.append(dict(entry, reason=f"not on {mount}"))
            continue
        resolved.append(scene)
    return resolved, unresolved


# --- what is mounted ------------------------------------------------------
if KAGGLE_INPUT.is_dir():
    MOUNTS = sorted(p for p in KAGGLE_INPUT.iterdir() if p.is_dir())
    log.info("%d dataset(s) mounted at %s", len(MOUNTS), KAGGLE_INPUT)
    for mount in MOUNTS:
        n_files = sum(1 for p in mount.rglob("*") if p.is_file())
        log.info("  %-44s %6d files", mount.name, n_files)
    if not MOUNTS:
        log.warning("nothing attached: Kaggle sidebar > Input > Add Input")
else:
    MOUNTS = []
    log.info("no %s (not on Kaggle); using DATA_ROOT=%s", KAGGLE_INPUT, DATA_ROOT)

# --- ground truth on demand -----------------------------------------------
DTU_STL_RE = re.compile(r"stl(\d+)_total\.ply$", re.IGNORECASE)
# Block size here is a latency trade, not a bandwidth one. DTU's server is in
# Denmark and every block is a fresh HTTPS request, so at 1 MB a block a 2 MB
# frame costs three round trips: the image fetch measured 0.7 MB/s from Kaggle,
# 26 min for ten scans, while the reference clouds -- read as one long stream --
# managed 1.9 MB/s over the same link. At 8 MB one request serves three or four
# frames. Eight blocks cached is 64 MB of RAM, and reads here are sequential.
_RANGE_BLOCK = 8 << 20
_RANGE_CACHE_BLOCKS = 8


def _ranged_get(url, start=None, stop=None):
    # DTU_POINTS_URL is editable config, so the scheme is checked before the
    # open: a file:// or custom scheme must not turn a download into a local
    # read. That check is what makes the urlopen below safe (ruff S310).
    if not str(url).lower().startswith(("http://", "https://")):
        raise ValueError(f"refusing to fetch a non-HTTP url: {url}")
    span = "bytes=0-0" if start is None else f"bytes={start}-{stop}"
    request = urllib.request.Request(url, headers={"Range": span})  # noqa: S310
    return urllib.request.urlopen(request)  # noqa: S310


class _HttpRangeFile(io.RawIOBase):
    """Seekable read-only view of a remote file, backed by HTTP Range requests.

    zipfile needs only seek and read, so pointing it at one of these reads a
    remote archive's central directory and then the handful of members we ask
    for, leaving the other 6 GB on the server.
    """

    def __init__(self, url):
        self.url = url
        self.pos = 0
        self._blocks, self._order = {}, []
        # A ranged GET rather than HEAD: it reports the total in Content-Range
        # and proves the server honours ranges, in one request.
        with _ranged_get(url) as response:
            content_range = response.headers.get("Content-Range")
        if not content_range or "/" not in content_range:
            raise RuntimeError(f"{url} does not serve byte ranges")
        self.size = int(content_range.rsplit("/", 1)[1])

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        base = {io.SEEK_SET: 0, io.SEEK_CUR: self.pos, io.SEEK_END: self.size}[whence]
        self.pos = max(0, min(self.size, base + offset))
        return self.pos

    def _block(self, index):
        cached = self._blocks.get(index)
        if cached is not None:
            return cached
        start = index * _RANGE_BLOCK
        stop = min(start + _RANGE_BLOCK, self.size) - 1
        with _ranged_get(self.url, start, stop) as response:
            data = response.read()
        self._blocks[index] = data
        self._order.append(index)
        if len(self._order) > _RANGE_CACHE_BLOCKS:
            self._blocks.pop(self._order.pop(0), None)
        return data

    def read(self, size=-1):
        if size < 0:
            size = self.size - self.pos
        size = min(size, self.size - self.pos)
        out = bytearray()
        while size > 0:
            index, offset = divmod(self.pos, _RANGE_BLOCK)
            chunk = self._block(index)[offset : offset + size]
            if not chunk:
                break
            out += chunk
            self.pos += len(chunk)
            size -= len(chunk)
        return bytes(out)

    def readinto(self, buffer):
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def fetch_dtu_points(scans, url, dest):
    # Writes <dest>/stl/stl0NN_total.ply, the layout _match_gt already pairs
    # with Rectified/scanNN by scan number.
    stl_dir = Path(dest) / "stl"
    stl_dir.mkdir(parents=True, exist_ok=True)
    wanted = {int(scan): stl_dir / f"stl{int(scan):03d}_total.ply" for scan in scans}
    missing = {scan: path for scan, path in wanted.items() if not path.exists()}
    if not missing:
        log.info("DTU reference clouds already cached at %s", stl_dir)
        return Path(dest)

    log.info("fetching %d DTU reference cloud(s) by byte range from %s", len(missing), url)
    with zipfile.ZipFile(_HttpRangeFile(url)) as archive:
        members = {}
        for info in archive.infolist():
            match = DTU_STL_RE.search(info.filename)
            if match:
                members[int(match.group(1))] = info
        absent = sorted(set(missing) - set(members))
        if absent:
            # Warned, not fatal: a scan with no reference cloud is a scene
            # B2/B3/B4/B6 skip, and the rest of the lane still runs.
            log.warning("no reference cloud for scan(s) %s in %s", absent, url)
        for scan, target in sorted(missing.items()):
            info = members.get(scan)
            if info is None:
                continue
            # Write to .part and rename: a session killed mid-transfer must not
            # leave a truncated cloud that looks complete to the next run.
            part = target.with_suffix(".part")
            with archive.open(info) as src, open(part, "wb") as out:
                shutil.copyfileobj(src, out, 1 << 20)
            part.replace(target)
            log.info("  %-24s %7.1f MB", target.name, target.stat().st_size / 1024**2)
    _invalidate_tree(Path(dest))
    return Path(dest)


# Rectified/scanNN/rect_001_3_r5000.png -> (24, "rect_001_3_r5000.png")
DTU_RECT_RE = re.compile(r"(?:^|/)scan(\d+)/([^/]+)$", re.IGNORECASE)


def fetch_dtu_images(scans, url, dest, glob="*"):
    """Pull one lighting condition per scan out of Rectified.zip by byte range.

    The same trick as fetch_dtu_points against a much larger archive: 129.6 GB
    for 124 scans at seven lighting conditions, of which one condition is
    77-146 MB a scan (49 frames), plus ~7 MB to read the central directory.
    Writes <dest>/Rectified/scanNN/, which is the layout the image_dir hints in
    KAGGLE_SCENES already name.
    """
    root = Path(dest)
    wanted = {int(scan): root / "Rectified" / f"scan{int(scan)}" for scan in scans}
    missing = {
        scan: directory
        for scan, directory in wanted.items()
        if len(_images_in(directory, glob)) < MIN_SCENE_IMAGES
    }
    if not missing:
        log.info("DTU images already cached at %s", root / "Rectified")
        return root

    log.info("fetching %d DTU scan(s) matching %s by byte range from %s", len(missing), glob, url)
    with zipfile.ZipFile(_HttpRangeFile(url)) as archive:
        members = {}
        for info in archive.infolist():
            match = DTU_RECT_RE.search(info.filename)
            if match is None:
                continue
            scan = int(match.group(1))
            if scan in missing and fnmatch.fnmatch(match.group(2), glob):
                members.setdefault(scan, []).append(info)
        absent = sorted(set(missing) - set(members))
        if absent:
            # Warned, not fatal. This is also what a lighting glob that matches
            # nothing looks like, so it names the glob as well as the scans.
            log.warning(
                "nothing matching %s for scan(s) %s in %s -- those scenes are dropped",
                glob,
                absent,
                url,
            )
        for scan, infos in sorted(members.items()):
            target_dir = missing[scan]
            target_dir.mkdir(parents=True, exist_ok=True)
            written = 0
            for info in sorted(infos, key=lambda item: item.filename):
                target = target_dir / info.filename.rsplit("/", 1)[-1]
                if target.exists():
                    continue
                # .part then rename, as above: a session killed mid-transfer
                # must not leave a truncated frame that decodes far enough to
                # reach MASt3R.
                part = target.with_suffix(".part")
                with archive.open(info) as src, open(part, "wb") as out:
                    shutil.copyfileobj(src, out, 1 << 20)
                part.replace(target)
                written += target.stat().st_size
            log.info("  scan%-5d %4d frame(s)  %7.1f MB", scan, len(infos), written / 1024**2)
    _invalidate_tree(root)
    return root


def attach_ground_truth(scenes):
    # Runs after the fetch, so the index it searches includes what this
    # notebook just downloaded. A scene that still has no cloud keeps
    # gt_path=None and B2/B3/B4/B6 skip it -- section 10 says so per scene.
    gt_files = gt_files_everywhere()
    for scene in scenes:
        if scene.get("gt_path"):
            continue
        candidate = _match_gt(scene["name"], Path(scene["image_dir"]), gt_files)
        if candidate is None and len(gt_files) == 1:
            candidate = gt_files[0]
        if candidate is None:
            continue
        scene["gt_path"] = str(candidate)
        scan = _first_number(scene["name"])
        if DTU_STL_RE.search(candidate.name) and scan == _first_number(candidate.stem):
            # DTU's own structured-light cloud for this scan, matched by number:
            # sensor ground truth, not a stand-in for one.
            log.info("%s: ground truth %s", scene["name"], candidate.name)
        else:
            log.warning(
                "%s: no gt_path configured; taking %s as pseudo-GT. Verify it "
                "before reporting any F-score against it.",
                scene["name"],
                candidate,
            )


# --- resolve the scene list -----------------------------------------------
# Images first, ground truth second. The reference clouds are fetched only for
# the scans whose photographs actually turned up, so a scan that is about to be
# dropped does not cost a 122 MB download on its way out.
RESOLVED_SCENES, MISSING_SCENES = resolve_configured(SCENES, DATA_ROOT)
_kaggle_scenes, UNRESOLVED_ENTRIES = resolve_kaggle_scenes(KAGGLE_SCENES)
RESOLVED_SCENES.extend(_kaggle_scenes)

# --- images on demand -----------------------------------------------------
if UNRESOLVED_ENTRIES and DTU_IMAGES_URL:
    _wanted = set(DTU_SCANS)
    _scans = sorted(
        {
            scan
            for scan in (_first_number(entry["name"]) for entry in UNRESOLVED_ENTRIES)
            if scan in _wanted
        }
    )
    if _scans:
        for _entry in UNRESOLVED_ENTRIES:
            log.info("%s unresolved on a mount (%s)", _entry["name"], _entry["reason"])
        _images_root = fetch_dtu_images(
            _scans, DTU_IMAGES_URL, WORK_DIR / "dtu_images", DTU_LIGHTING_GLOB
        )
        FETCHED_ROOTS.append(_images_root)
        _still_missing = []
        for _entry in UNRESOLVED_ENTRIES:
            _scene = resolve_entry(_entry, _images_root, source=f"fetched: {DTU_IMAGES_URL}")
            if _scene is None:
                _still_missing.append(_entry)
            else:
                RESOLVED_SCENES.append(_scene)
        UNRESOLVED_ENTRIES = _still_missing

MISSING_SCENES.extend(f"{entry['name']}: {entry['reason']}" for entry in UNRESOLVED_ENTRIES)
for miss in MISSING_SCENES:
    log.warning("scene not resolved, skipped: %s", miss)

# Configured order, whatever route each scene arrived by. The smoke lane runs
# the first scene and FULL_SCENE_LIMIT takes the first N, so "first" has to
# mean the config's first rather than whichever mount answered soonest.
_ORDER = {entry["name"]: position for position, entry in enumerate([*SCENES, *KAGGLE_SCENES])}
RESOLVED_SCENES.sort(key=lambda scene: _ORDER.get(scene["name"], len(_ORDER)))

# --- ground truth on demand -----------------------------------------------
if DTU_POINTS_URL and DTU_SCANS:
    _wanted = set(DTU_SCANS)
    _need_gt = {
        _first_number(scene["name"])
        for scene in RESOLVED_SCENES
        if not scene.get("gt_path") and _first_number(scene["name"]) in _wanted
    }
    _have = set()
    for _path in gt_files_everywhere():
        _match = DTU_STL_RE.search(_path.name)
        if _match:
            _have.add(int(_match.group(1)))
    _need = sorted(_need_gt - _have)
    if _need:
        log.info(
            "%d of %d resolved DTU scan(s) have no ground truth on any mount",
            len(_need),
            len(_need_gt),
        )
        FETCHED_ROOTS.append(fetch_dtu_points(_need, DTU_POINTS_URL, WORK_DIR / "dtu_points"))
        _GT_INDEX = None  # re-index now that the clouds are on disk
    elif _need_gt:
        log.info("every resolved DTU scan already has ground truth on a mount")

attach_ground_truth(RESOLVED_SCENES)

if not RESOLVED_SCENES and AUTO_DISCOVER_SCENES:
    # Every root, not just the first that yields something: a session with the
    # capture dataset and DTU both attached must not silently drop the one that
    # carries ground truth.
    seen_roots, seen_dirs, used_names = set(), set(), set()
    for search_root in (Path(DATA_ROOT), *FETCHED_ROOTS, *MOUNTS):
        if search_root in seen_roots or not search_root.is_dir():
            continue
        seen_roots.add(search_root)
        log.info("no configured scene resolved; walking %s", search_root)
        for scene in discover_scenes(
            search_root, MAX_DISCOVERED_SCENES - len(RESOLVED_SCENES), used_names
        ):
            if scene["image_dir"] in seen_dirs:
                continue
            seen_dirs.add(scene["image_dir"])
            RESOLVED_SCENES.append(scene)
        if len(RESOLVED_SCENES) >= MAX_DISCOVERED_SCENES:
            log.warning(
                "stopped at MAX_DISCOVERED_SCENES=%d; raise it to take in more",
                MAX_DISCOVERED_SCENES,
            )
            break

if not RESOLVED_SCENES:
    raise SystemExit(
        "no scenes. Attach a dataset (Kaggle sidebar > Input > Add Input), point "
        "DATA_ROOT at its mount and list the scenes in SCENES -- or leave "
        "AUTO_DISCOVER_SCENES on and let this cell find them."
    )

print(f"{'scene':<24} {'images':>7}  {'gt':<5}  image_dir")
print("-" * 96)
for scene in RESOLVED_SCENES:
    print(
        f"{scene['name']:<24} {scene.get('n_images', 0):>7}  "
        f"{'yes' if scene.get('gt_path') else 'NO':<5}  {scene['image_dir']}"
    )
for scene in RESOLVED_SCENES:
    dropped = _n_dropped(scene["image_dir"], scene.get("image_glob"))
    if dropped:
        log.warning(
            "%s: excluded %d depth/normal/mask file(s) that a suffix filter alone "
            "would have fed to MASt3R as photographs",
            scene["name"],
            dropped,
        )

# Scans that never resolved. Warned rather than fatal -- the lane still runs,
# it just runs on fewer objects than the config asked for, and the aggregate
# has to say so.
DROPPED_SCANS = sorted(set(DTU_SCANS) - {_first_number(scene["name"]) for scene in RESOLVED_SCENES})
if DROPPED_SCANS:
    log.warning(
        "%d of %d requested scan(s) are NOT in this lane: %s. Nothing is "
        "reconstructed or scored on them.",
        len(DROPPED_SCANS),
        len(DTU_SCANS),
        DROPPED_SCANS,
    )

N_WITH_GT = sum(1 for s in RESOLVED_SCENES if s.get("gt_path"))
print(f"\n{len(RESOLVED_SCENES)} scene(s), {N_WITH_GT} with ground truth")
if N_WITH_GT == 0:
    log.warning(
        "no scene has ground truth. Both B2 and B4 skip scenes with gt_path=None "
        "and will produce ZERO ROWS -- this lane has nothing to run without it. "
        "Set DTU_POINTS_URL (or attach a Points/stl mirror) before spending a "
        "full lane on this."
    )

## 9. Scene manifest

Written into the work dir with **absolute** paths, so a read-only
`/kaggle/input` mount works and nothing has to be copied. Only the manifest
keys are carried over: discovery and the dataset test hang working state off
the same dicts.

In [ ]:
MANIFEST_PATH = WORK_DIR / "scenes.json"

# Discovery and the dataset test hang working state (frame counts, loaded
# paths) off the same scene dicts, so the manifest takes named keys only.
MANIFEST_KEYS = ("name", "image_dir", "gt_path", "image_glob", "tau", "units", "notes")

manifest = {
    "_generated_by": "tier_b_gpu.ipynb run " + RUN_ID,
    "tau": DEFAULT_TAU,
    "units": DEFAULT_UNITS,
    "scenes": [
        {key: scene[key] for key in MANIFEST_KEYS if key in scene} for scene in RESOLVED_SCENES
    ],
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
log.info("manifest -> %s (%d scenes)", MANIFEST_PATH, len(manifest["scenes"]))
print(MANIFEST_PATH.read_text())

## 10. Dataset test

A pytest-shaped pass over the mounted data, in this kernel, in seconds. It runs
*before* the checkpoint download and the GPU work because every failure below is
cheap here and expensive later.

| Check | Fails when | Why it is here |
|---|---|---|
| `images` | fewer than 2 images | MASt3R needs a pair. Warns at <= 40 frames, where B4's budget never fires and its three variants come out identical |
| `decode` | any file fails its header check, or a sampled full decode comes back `None` | every image is header-checked and up to 8 are fully decoded; one corrupt frame takes a run down in the middle of the grid. Warns on mixed resolutions inside one scene |
| `capture_order` | *(warn only)* | `Scene.image_paths` sorts lexically and every B module reads that as capture order; unpadded numbering silently reverses it (`frame10` before `frame2`) -- the defect A5 measured |
| `ground_truth` | trimesh cannot load it, fewer than 1000 points, or non-finite coordinates | loaded through `bench.tier_b_common.load_gt_points`, so a pass here is a pass in B2 |
| `tau_vs_gt_scale` | *(warn only)* | `tau` outside 0.01%-10% of the GT bounding diagonal means the manifest and the GT file disagree about units. A metres-vs-mm mix-up yields a perfectly plausible F-score that means nothing |
| `input_mount` | no scene came from a mount *or* a fetch | catches a notebook pointed at a dataset that was never added. A lane that resolved entirely through `DTU_IMAGES_URL` passes |
| `requested_scans` | *(warn only)* | a scan in `DTU_SCANS` that never resolved is a hole in the aggregate: nothing is reconstructed or scored on it, and the row count drops without the CSV saying why |
| `manifest_roundtrip` | `SceneSet.from_manifest` changes the scene list, or `image_paths` comes back unsorted | reads the manifest just written back through the exact class B1-B8 use |

Statuses are `ok` / `WARN` (read it, keep going) / `skip` (could not run) /
`FAIL`. Failures stop the notebook while `DATASET_TEST_STRICT` is on. The table
truncates each detail; the full record goes to
`logs/<run_id>/dataset_test.json`.

In [ ]:
import functools

import numpy as np

# Import the repo into this kernel so the checks use exactly the loaders Tier B
# uses: a ground-truth file trimesh cannot open here will not open in B2 either.
for _path in (str(REPO_DIR), str(REPO_DIR / "src")):
    if _path not in sys.path:
        sys.path.insert(0, _path)

try:
    from bench.tier_b_common import SceneSet, load_gt_points
    from spatial_ingestion.config import MAX_RECONSTRUCTION_FRAMES
except Exception as exc:
    SceneSet, load_gt_points, MAX_RECONSTRUCTION_FRAMES = None, None, 40
    log.warning("bench not importable in this kernel (%r); GT checks will be skipped", exc)


class CheckSkipped(Exception):
    """The check could not run: no ground truth, no EXIF, no bench import."""


class CheckWarning(Exception):
    """It ran and found something you should read, but not a reason to stop."""


# Header-check every image; fully decode at most this many, evenly spaced.
MAX_FULL_DECODES = 8

RESULTS = []


def check_case(scene_name, check_name, fn):
    try:
        detail, status = fn(), "pass"
    except CheckSkipped as exc:
        detail, status = str(exc), "skip"
    except CheckWarning as exc:
        detail, status = str(exc), "warn"
    except Exception as exc:
        detail, status = f"{type(exc).__name__}: {exc}", "fail"
        log.exception("FAIL  %s / %s", scene_name, check_name)
    RESULTS.append(
        {"scene": scene_name, "check": check_name, "status": status, "detail": str(detail)}
    )
    return status


# --- per-scene checks -----------------------------------------------------
def _check_images(scene):
    paths = _images_in(scene["image_dir"], scene.get("image_glob"))
    scene["_paths"] = paths
    if len(paths) < 2:
        raise RuntimeError(f"{len(paths)} image(s) -- MASt3R needs a pair at minimum")
    if len(paths) <= MAX_RECONSTRUCTION_FRAMES:
        raise CheckWarning(
            f"{len(paths)} frames <= MAX_RECONSTRUCTION_FRAMES={MAX_RECONSTRUCTION_FRAMES}: "
            "the budget never fires, so B4's three variants come out identical"
        )
    return f"{len(paths)} images"


def _check_decode(scene):
    # Every file gets a header check: one corrupt frame takes a run down in the
    # middle of the grid, and catching it here costs a second. A spaced sample
    # is fully decoded on top -- that is what catches truncation past the
    # header, and it is where the resolution comes from.
    import cv2
    from PIL import Image

    paths = scene.get("_paths") or []
    if not paths:
        raise CheckSkipped("no images")

    unreadable = []
    for path in paths:
        try:
            with Image.open(path) as image:
                image.verify()
        except Exception as exc:
            unreadable.append(f"{path.name} ({type(exc).__name__})")
    if unreadable:
        raise RuntimeError(f"{len(unreadable)} unreadable file(s): " + ", ".join(unreadable[:4]))

    sample = paths[:: max(1, len(paths) // MAX_FULL_DECODES)][:MAX_FULL_DECODES]
    shapes = []
    for path in sample:
        image = cv2.imread(str(path))
        if image is None:
            raise RuntimeError(f"cv2 returned None for {path.name} -- truncated, or an odd depth")
        shapes.append(image.shape[:2])
    scene["_resolution"] = shapes[0]
    if len(set(shapes)) > 1:
        raise CheckWarning(
            "mixed resolutions inside one scene: "
            + ", ".join(f"{w}x{h}" for h, w in sorted(set(shapes)))
        )
    height, width = shapes[0]
    return f"{width}x{height}, {len(paths)} headers + {len(sample)} full decodes"


def _check_order(scene):
    # Scene.image_paths sorts lexically and every B module treats that as
    # capture order. Unpadded numbering breaks it silently (frame10 sorts before
    # frame2) -- the same ordering defect A5 measured on the CPU side.
    #
    # Compare every numeric field, not just one: DTU's rect_001_0_r5000.png
    # carries the frame index first and a lighting index last, so reading a
    # single field passes vacuously on the wrong number.
    paths = scene.get("_paths") or []
    if not paths:
        raise CheckSkipped("no images")
    keys = [tuple(int(number) for number in re.findall(r"\d+", path.stem)) for path in paths]
    if not all(keys):
        raise CheckWarning("some filenames carry no number; capture order is the filesystem's")
    if len({len(key) for key in keys}) > 1:
        raise CheckWarning("filenames do not share one numbering scheme; check the capture order")
    if keys != sorted(keys):
        first_bad = next(i for i in range(1, len(keys)) if keys[i] < keys[i - 1])
        raise CheckWarning(
            f"lexical order is not numeric order ({paths[first_bad - 1].name} sorts before "
            f"{paths[first_bad].name}): zero-pad the numbers, or B4 reconstructs out of order"
        )
    return f"{len(paths)} names in numeric order, {len(keys[0])} numeric field(s)"


def _check_gt(scene):
    gt_path = scene.get("gt_path")
    if not gt_path:
        raise CheckSkipped("gt_path is null -- B2/B4 skip this scene")
    if load_gt_points is None:
        raise CheckSkipped("bench.tier_b_common not importable in this kernel")
    points = load_gt_points(Path(gt_path), max_points=50000)
    if len(points) < 1000:
        raise RuntimeError(f"only {len(points)} ground-truth points")
    if not np.isfinite(points).all():
        raise RuntimeError("ground truth contains non-finite coordinates")
    extent = points.max(axis=0) - points.min(axis=0)
    scene["_gt_diagonal"] = float(np.linalg.norm(extent))
    return (
        f"{len(points)} pts, bbox {extent[0]:.3g} x {extent[1]:.3g} x {extent[2]:.3g} "
        f"{scene.get('units', '?')}"
    )


def _check_tau(scene):
    # tau stated in the wrong unit is the failure that produces a plausible
    # F-score meaning nothing. DTU's 2 mm against a ~300 mm object is 0.7% of
    # the bounding diagonal; outside 0.01%-10% the manifest and the GT file
    # almost certainly disagree about units.
    diagonal = scene.get("_gt_diagonal")
    if diagonal is None:
        raise CheckSkipped("ground truth not loaded")
    tau = float(scene.get("tau", DEFAULT_TAU))
    ratio = tau / diagonal
    if scene.get("tau_is_default"):
        # A plausible-looking ratio is not evidence: DEFAULT_TAU was never
        # chosen for this dataset, and the units label came from the config,
        # not the file. An F-score computed against this is arbitrary.
        raise CheckWarning(
            f"tau={tau} is DEFAULT_TAU, and this scene's units are unknown "
            f"({ratio:.3%} of the GT diagonal {diagonal:.4g}). Set tau and units "
            "explicitly in SCENES/KAGGLE_SCENES before reporting any F-score."
        )
    if not 1e-4 <= ratio <= 1e-1:
        raise CheckWarning(
            f"tau={tau} {scene.get('units', '?')} is {ratio:.3%} of the GT bounding diagonal "
            f"({diagonal:.4g}) -- check the units before trusting any F-score"
        )
    return f"tau is {ratio:.3%} of the GT diagonal"


# --- dataset-wide checks --------------------------------------------------
def _check_mount():
    # Attaching a dataset is one way in; the byte-range fetch is the other, and
    # a lane that resolved entirely through the fetch is not a broken one.
    if not KAGGLE_INPUT.is_dir():
        raise CheckSkipped(f"{KAGGLE_INPUT} does not exist -- not running on Kaggle")
    used = [
        mount
        for mount in MOUNTS
        if any(scene["image_dir"].startswith(str(mount)) for scene in RESOLVED_SCENES)
    ]
    fetched = sum(
        1
        for scene in RESOLVED_SCENES
        if any(scene["image_dir"].startswith(str(root)) for root in FETCHED_ROOTS)
    )
    if not used and not fetched:
        raise RuntimeError(
            "no scene came from a mount or from a fetch: attach a dataset "
            "(Kaggle sidebar > Input > Add Input), or set DTU_IMAGES_URL"
        )
    detail = f"{len(used)} of {len(MOUNTS)} mount(s) in use"
    if used:
        detail += ": " + ", ".join(m.name for m in used)
    if fetched:
        detail += f"; {fetched} scene(s) fetched into {WORK_DIR}"
    return detail


def _check_requested_scans():
    # DTU_SCANS drives the fetch and KAGGLE_SCENES both, so a scan that never
    # resolved is a hole in the aggregate. Warned rather than failed: the lane
    # on the scans that did resolve is still worth running.
    if not DTU_SCANS:
        raise CheckSkipped("DTU_SCANS is empty")
    if DROPPED_SCANS:
        raise CheckWarning(
            f"{len(DROPPED_SCANS)} of {len(DTU_SCANS)} requested scan(s) never resolved and "
            f"are not in this lane: {DROPPED_SCANS}"
        )
    return f"all {len(DTU_SCANS)} requested scan(s) resolved"


def _check_manifest():
    if SceneSet is None:
        raise CheckSkipped("bench.tier_b_common not importable in this kernel")
    scene_set = SceneSet.from_manifest(MANIFEST_PATH)
    names = [scene.name for scene in scene_set.scenes]
    if names != [scene["name"] for scene in RESOLVED_SCENES]:
        raise RuntimeError(f"round-trip changed the scene list: {names}")
    for scene in scene_set.scenes:
        wanted = min(4, len(_images_in(scene.image_dir, scene.image_glob)))
        paths = scene.image_paths(limit=wanted)
        if len(paths) != wanted:
            raise RuntimeError(f"{scene.name}: image_paths(limit={wanted}) returned {len(paths)}")
        if paths != sorted(paths):
            raise RuntimeError(f"{scene.name}: image_paths did not come back sorted")
    return f"{len(names)} scene(s) survived SceneSet.from_manifest"


SCENE_CHECKS = (
    ("images", _check_images),
    ("decode", _check_decode),
    ("capture_order", _check_order),
    ("ground_truth", _check_gt),
    ("tau_vs_gt_scale", _check_tau),
)

log.info("dataset test: %d scene(s) x %d checks", len(RESOLVED_SCENES), len(SCENE_CHECKS))
for scene in RESOLVED_SCENES:
    for check_name, check_fn in SCENE_CHECKS:
        check_case(scene["name"], check_name, functools.partial(check_fn, scene))
check_case("-", "input_mount", _check_mount)
check_case("-", "requested_scans", _check_requested_scans)
check_case("-", "manifest_roundtrip", _check_manifest)

MARK = {"pass": "ok", "warn": "WARN", "fail": "FAIL", "skip": "skip"}
print()
print("=" * 100)
print(f"{'scene':<18} {'check':<18} {'':<4} detail")
print("-" * 100)
for row in RESULTS:
    print(f"{row['scene']:<18} {row['check']:<18} {MARK[row['status']]:<4} {row['detail'][:56]}")
print("=" * 100)

TALLY = {status: sum(1 for row in RESULTS if row["status"] == status) for status in MARK}
print(
    f"{TALLY['pass']} passed, {TALLY['warn']} warned, {TALLY['skip']} skipped, "
    f"{TALLY['fail']} failed"
)

DATASET_TEST_PATH = LOG_DIR / "dataset_test.json"
DATASET_TEST_PATH.write_text(
    json.dumps(
        {
            "run_id": RUN_ID,
            "data_root": str(DATA_ROOT),
            "manifest": str(MANIFEST_PATH),
            "tally": TALLY,
            "results": RESULTS,
        },
        indent=2,
    ),
    encoding="utf-8",
)
log.info("dataset test -> %s (full detail; the table above is truncated)", DATASET_TEST_PATH)

if TALLY["fail"] and DATASET_TEST_STRICT:
    raise SystemExit(
        f"{TALLY['fail']} dataset check(s) failed. Fix the data before spending GPU hours "
        f"on it, or set DATASET_TEST_STRICT = False to proceed anyway. "
        f"Detail: {DATASET_TEST_PATH}"
    )

## 11. Preflight

A hard gate. Everything that can fail cheaply fails here, before hours of
compute: imports, CUDA, checkpoint download, every scene path, every GT file,
frame counts, disk. Fatal problems raise; the rest are warnings you should read.

In [ ]:
FATAL = []
WARN = []


def check(label, fn, fatal=True):
    try:
        result = fn()
    except Exception as exc:
        (FATAL if fatal else WARN).append(f"{label}: {exc!r}")
        log.exception("FAIL  %s", label)
        return None
    log.info("ok    %-34s %s", label, "" if result is None else result)
    return result


def _imports():
    mods = (
        "torch",
        "numpy",
        "cv2",
        "trimesh",
        "pyvista",
        "scipy",
        "sklearn",
        "roma",
        "einops",
        "mast3r.model",
        "dust3r.utils.image",
        "spatial_ingestion.reconstruction.pipeline",
        "spatial_ingestion.final_pipeline.handoff",
        "bench.tier_b_common",
    )
    src = (
        "import importlib\n"
        + "\n".join(f"importlib.import_module({m!r})" for m in mods)
        + "\nprint('imports ok')"
    )
    sh(
        [sys.executable, "-c", src],
        cwd=REPO_DIR,
        env=CHILD_ENV,
        log_path=LOG_DIR / "preflight.log",
        echo=False,
        timeout_s=600,
    )
    return f"{len(mods)} modules"


def _cuda():
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "torch.cuda.is_available() is False. Enable a GPU accelerator "
            "(Kaggle: Settings > Accelerator; Colab: Runtime > Change runtime type)."
        )
    free, total = torch.cuda.mem_get_info(0)
    name = torch.cuda.get_device_name(0)
    return f"{name}, {free / 1024**3:.1f}/{total / 1024**3:.1f} GB free"


def _checkpoint():
    # Download the weights here, where a network failure costs seconds.
    src = (
        "import time\n"
        "from mast3r.model import AsymmetricMASt3R\n"
        "t = time.time()\n"
        "m = AsymmetricMASt3R.from_pretrained("
        "'naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric')\n"
        "m = m.to('cuda').eval()\n"
        "n = sum(p.numel() for p in m.parameters())\n"
        "print('loaded %.0fM params to cuda in %.1fs' % (n / 1e6, time.time() - t))\n"
    )
    _, tail = sh(
        [sys.executable, "-c", src],
        cwd=REPO_DIR,
        env=CHILD_ENV,
        log_path=LOG_DIR / "preflight.log",
        timeout_s=1800,
        echo=False,
    )
    lines = [ln for ln in tail.strip().splitlines() if ln.strip()]
    return lines[-1] if lines else "loaded"


def _scenes():
    entries = json.loads(MANIFEST_PATH.read_text())["scenes"]
    if not entries:
        raise RuntimeError("manifest has no scenes")
    summary = []
    for entry in entries:
        image_dir = Path(entry["image_dir"])
        if not image_dir.is_dir():
            raise FileNotFoundError(f"{entry['name']}: image_dir does not exist: {image_dir}")
        # _images_in, not a bare suffix count: the preflight frame count has to
        # be the number the run will actually reconstruct.
        n = len(_images_in(image_dir, entry.get("image_glob")))
        if n == 0:
            raise FileNotFoundError(f"{entry['name']}: no images under {image_dir}")
        gt = entry.get("gt_path")
        if not gt:
            WARN.append(
                f"{entry['name']}: gt_path is null. B2/B4 skip scenes with no GT -- "
                "this scene contributes zero rows."
            )
        elif not Path(gt).exists():
            raise FileNotFoundError(f"{entry['name']}: gt_path missing: {gt}")
        if n <= 40:
            WARN.append(
                f"{entry['name']}: only {n} frames. B4 caps at MAX_RECONSTRUCTION_FRAMES=40, "
                "so the budget never fires and its three variants are identical."
            )
        summary.append(f"{entry['name']}={n} frames")
    return ", ".join(summary)


def _disk():
    problems = []
    for label, path, need_gb in (("work", WORK_DIR, 25), ("out", OUT_DIR, 2)):
        free = shutil.disk_usage(path).free / 1024**3
        if free < need_gb:
            problems.append(f"{label} has {free:.1f} GB free, want >= {need_gb} GB")
    if problems:
        raise RuntimeError("; ".join(problems))
    return "sufficient"


check("imports", _imports)
check("cuda", _cuda)
check("scenes", _scenes)
check("disk", _disk)
check("mast3r checkpoint", _checkpoint)

print()
for w in WARN:
    log.warning("WARN  %s", w)
if FATAL:
    for f in FATAL:
        log.error("FATAL %s", f)
    raise SystemExit(
        f"{len(FATAL)} preflight failure(s). Fix these before running anything long. "
        f"Full detail: {LOG_DIR / 'preflight.log'}"
    )
log.info("preflight passed (%d warnings)", len(WARN))

## 12. Experiment table

`--quick` is deliberately never passed: no `exp_b*.py` module forwards it to
`run()`, so it would do nothing while looking like it did something. Scope comes
from the flags each module actually reads.

Note B4: its frame budget is the module constant `DEFAULT_BUDGET = 40` with no
CLI override, so even the smoke lane runs three 40-frame reconstructions. It is
the longest smoke item by a wide margin, which is why it sits late in
`EXPERIMENTS`. Wiring `--quick` into `run()` is the real fix.

In [ ]:
import math

# Trimmed to the two experiments this lane runs (see the config cell). The
# other six modules still exist in bench/ -- add their entries back here to
# re-enable them.
MODULES = {
    "b2": ("b2_reconstruction_accuracy", "bench.exp_b2_reconstruction_accuracy"),
    "b4": ("b4_frame_budget_ablation", "bench.exp_b4_frame_budget_ablation"),
}

FIRST_SCENE = manifest["scenes"][0]["name"]
ALL_SCENES = [scene["name"] for scene in manifest["scenes"]]


def scenes_for(exp, scope=None):
    # The smoke lane is one scene by construction. In the full lane an
    # experiment absent from FULL_SCENE_LIMIT gets every scene there is.
    scope = scope or SCOPE
    if scope == "smoke":
        return [FIRST_SCENE]
    limit = FULL_SCENE_LIMIT.get(exp)
    return ALL_SCENES if limit is None else ALL_SCENES[:limit]


def timeout_min_for(exp, scope=None):
    # TIMEOUT_MIN's full-lane numbers are PER SCENE (see the config cell), so a
    # tuple of ten scans does not silently turn every ceiling into a bug report.
    scope = scope or SCOPE
    minutes = TIMEOUT_MIN[scope][exp]
    return minutes if scope == "smoke" else minutes * max(1, len(scenes_for(exp, scope)))


SMOKE_ARGS = {
    "b2": ["--scenes", FIRST_SCENE, "--n-images", "4"],
    "b4": ["--scenes", FIRST_SCENE, "--no-budget-sweep"],
}

# Full lane: module defaults, every scene. Override here if the protocol changed.
FULL_ARGS = {key: [] for key in MODULES}

# FULL_SCENE_LIMIT, turned into the flag the modules actually read. Passed last:
# --scenes is nargs="*", so anything after it would be eaten as a scene name.
for _exp in MODULES:
    _capped = scenes_for(_exp, "full")
    if len(_capped) < len(ALL_SCENES):
        FULL_ARGS[_exp] = [*FULL_ARGS[_exp], "--scenes", *_capped]

# b2 and b4 both take --output-root, which keeps the ~1.5 GB per-run alignment
# cache off the persisted output volume. (b1's --deliverables-root doesn't
# apply here -- it isn't in MODULES.)
OUTPUT_FLAG = {}


def build_argv(exp, scope):
    _, module = MODULES[exp]
    args = (SMOKE_ARGS if scope == "smoke" else FULL_ARGS)[exp]
    out_root = WORK_DIR / "runs" / exp
    out_root.mkdir(parents=True, exist_ok=True)
    return [
        sys.executable,
        "-u",
        "-m",
        module,
        "--manifest",
        str(MANIFEST_PATH),
        "--results-dir",
        str(RESULTS_DIR),
        "--seed",
        "0",
        "--verbose",
        OUTPUT_FLAG.get(exp, "--output-root"),
        str(out_root),
        *args,
    ]


print(f"{'exp':<5} {'scenes':>6} {'ceiling':>9}  argv")
print("-" * 108)
for exp in EXPERIMENTS:
    print(
        f"{exp:<5} {len(scenes_for(exp)):>6} {timeout_min_for(exp):>7} m  "
        + " ".join(str(a) for a in build_argv(exp, SCOPE)[3:])
    )

# What this lane costs. Cell 1's 9-15 h describes ONE scene through all eight
# experiments; every experiment loops scenes, so the lane is linear in them.
SCENE_EXPERIMENTS = sum(len(scenes_for(exp)) for exp in EXPERIMENTS)
if SCOPE == "full":
    _factor = SCENE_EXPERIMENTS / len(MODULES)
    _low, _high = 9 * _factor, 15 * _factor
    _session_h = SESSION_BUDGET_MIN / 60
    print()
    print(
        f"{SCENE_EXPERIMENTS} scene-experiments over {len(ALL_SCENES)} scene(s) "
        f"({len(MODULES)} would be one scene through all eight), so about {_factor:.1f}x "
        f"cell 1's 9-15 h: roughly {_low:.0f}-{_high:.0f} h, or "
        f"{math.ceil(_low / _session_h)}-{math.ceil(_high / _session_h)} session(s) at "
        f"SESSION_BUDGET_MIN={SESSION_BUDGET_MIN} min. Kaggle's GPU quota is ~30 h/week."
    )
    print(
        "Trim DTU_SCANS or lower FULL_SCENE_LIMIT to shorten it. The ceilings above are "
        "ceilings, not estimates -- a run that reaches one is a bug, not progress."
    )
    # Each experiment is one process and bench.harness.finish writes its CSV
    # once, at the end. An experiment that cannot finish inside a session
    # therefore leaves nothing behind, however many scenes it got through --
    # and re-running repeats the same doomed run. Splitting it by hand with
    # --scenes is not free either: b2 discards a warm-up reconstruction only
    # when it is given more than one scene, so per-scene runs would record a
    # cold first timing for every scene.
    _tight = [exp for exp in EXPERIMENTS if timeout_min_for(exp) > SESSION_BUDGET_MIN]
    if _tight:
        print()
        print(
            f"NOTE: ceilings for {', '.join(_tight)} exceed SESSION_BUDGET_MIN="
            f"{SESSION_BUDGET_MIN} min. Ceilings are generous, so this is not a "
            "prediction -- but an experiment that genuinely runs past the session "
            "budget writes no CSV at all, because the writer flushes once when it "
            "finishes. If one of these overruns, lower FULL_SCENE_LIMIT for it "
            "rather than re-running it unchanged."
        )

## 13. Runner

One subprocess per experiment. A CUDA OOM, an OOM-killer `SIGKILL` or a VTK
segfault takes down that experiment only; the notebook records it and moves on.
`run_summary.json` is rewritten after each one, so a session killed at the wall
clock still leaves a complete account.

In [ ]:
SUMMARY_PATH = OUT_DIR / "run_summary.json"
RUNS = []
SESSION_START = time.monotonic()  # reset by section 14 before the lane starts
SESSION_START_ISO = _dt.datetime.now().isoformat(timespec="seconds")


def _save_summary():
    payload = {
        "run_id": RUN_ID,
        "platform": PLATFORM,
        "commit": COMMIT,
        "scope": SCOPE,
        "curope_built": CUROPE_BUILT,
        "started": SESSION_START_ISO,
        "updated": _dt.datetime.now().isoformat(timespec="seconds"),
        "runs": RUNS,
    }
    SUMMARY_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def _free_gb(path):
    return shutil.disk_usage(path).free / 1024**3


def _csv_rows(path):
    with path.open(encoding="utf-8") as fh:
        return max(0, sum(1 for _ in fh) - 1)


def run_experiment(exp, scope=None, force=None):
    scope = scope or SCOPE
    force = FORCE_RERUN if force is None else force
    exp_id, _ = MODULES[exp]
    csv_path = RESULTS_DIR / (exp_id + ".csv")
    log_path = LOG_DIR / (exp + ".log")

    if csv_path.exists() and not force:
        log.info("SKIP  %s -- %s exists (set FORCE_RERUN=True to redo)", exp, csv_path.name)
        record = {
            "exp": exp,
            "exp_id": exp_id,
            "status": "skipped",
            "minutes": 0.0,
            "rows": _csv_rows(csv_path),
            "csv": str(csv_path),
            "log": str(log_path),
        }
        RUNS.append(record)
        _save_summary()
        return record

    argv = build_argv(exp, scope)
    minutes = timeout_min_for(exp, scope)
    remaining = SESSION_BUDGET_MIN - (time.monotonic() - SESSION_START) / 60
    if 0 < remaining < minutes:
        # Capping keeps the notebook alive to record the failure and collect
        # the logs. It does not save the run: bench.harness.finish writes the
        # CSV once, at the end, so a cut experiment leaves no rows at all.
        log.warning(
            "%s: %d min ceiling is longer than the %.0f min left in the session "
            "budget -- capping there. The CSV is written only when an experiment "
            "finishes, so this run will leave no rows; lower FULL_SCENE_LIMIT or "
            "trim DTU_SCANS if it keeps happening",
            exp,
            minutes,
            remaining,
        )
        minutes = max(1, int(remaining))
    timeout_s = minutes * 60
    log.info("=" * 70)
    log.info(
        "START %s (%s lane, %d scene(s), timeout %d min)",
        exp,
        scope,
        len(scenes_for(exp, scope)),
        minutes,
    )
    log.info("      disk free: work %.1f GB, out %.1f GB", _free_gb(WORK_DIR), _free_gb(OUT_DIR))
    log.info("      log -> %s", log_path)

    started = time.monotonic()
    record = {
        "exp": exp,
        "exp_id": exp_id,
        "scope": scope,
        "scenes": scenes_for(exp, scope),
        "timeout_min": minutes,
        "argv": [str(a) for a in argv],
        "started": _dt.datetime.now().isoformat(timespec="seconds"),
        "log": str(log_path),
    }
    try:
        rc, _tail = sh(
            argv,
            cwd=REPO_DIR,
            env=CHILD_ENV,
            log_path=log_path,
            timeout_s=timeout_s,
            check=True,
            echo=True,
        )
        record["status"] = "ok"
        record["returncode"] = rc
    except CommandFailed as exc:
        record["status"] = "timeout" if exc.timed_out else "failed"
        record["returncode"] = exc.returncode
        record["error"] = exc.reason
        record["tail"] = exc.tail
        log.error("FAIL  %s: %s", exp, exc.reason)
        log.error("      last lines:\n%s", exc.tail)
    except Exception as exc:  # the runner itself broke
        record["status"] = "error"
        record["error"] = repr(exc)
        record["traceback"] = traceback.format_exc()
        log.exception("ERROR %s: runner failed", exp)

    record["minutes"] = round((time.monotonic() - started) / 60, 2)
    if csv_path.exists():
        record["csv"] = str(csv_path)
        record["rows"] = _csv_rows(csv_path)
        log.info("      %s: %d rows -> %s", exp, record["rows"], csv_path.name)
        if record["rows"] == 0:
            log.warning("      %s produced ZERO rows -- check gt_path and --scenes", exp)
    else:
        record["csv"] = None
        record["rows"] = 0
        if record["status"] == "ok":
            log.warning("      %s exited 0 but wrote no CSV", exp)

    log.info("DONE  %s: %s in %.1f min", exp, record["status"], record["minutes"])
    RUNS.append(record)
    _save_summary()

    if CLEAN_WORK_AFTER_EACH:
        target = WORK_DIR / "runs" / exp
        if target.exists():
            size_gb = sum(f.stat().st_size for f in target.rglob("*") if f.is_file()) / 1024**3
            shutil.rmtree(target, ignore_errors=True)
            log.info("      cleaned %.1f GB from %s", size_gb, target)
    return record

## 14. Run the tier

One cell per experiment below, instead of a single loop over `EXPERIMENTS`, so
a session that stalls, OOMs, or gets interrupted on one experiment leaves you
able to re-run just that block rather than the whole tier. Run "Start the
session" once, then the two experiment cells.

The cell order below (b2, then b4) runs the cheaper, faster-to-fail experiment
first, so a session that dies partway still bought you something. `EXPERIMENTS`
still controls *which* experiments run -- a block for an experiment missing
from it logs a skip and does nothing -- it does not control the order cells
execute in; reorder the cells by hand for a different sequence.

Each block stops itself (without launching) once `SESSION_BUDGET_MIN` is
reached, rather than starting something the platform's wall clock will cut
off. Re-run a block, or the whole notebook, to pick up where it stopped.

### Start the session

In [ ]:
SESSION_START = time.monotonic()
SESSION_START_ISO = _dt.datetime.now().isoformat(timespec="seconds")
RUNS.clear()

log.info("Tier B: %s lane, %d experiments, commit %s", SCOPE, len(EXPERIMENTS), COMMIT[:8])


def run_block(exp):
    if exp not in EXPERIMENTS:
        log.info("SKIP  %s -- not in EXPERIMENTS", exp)
        return
    elapsed_min = (time.monotonic() - SESSION_START) / 60
    if elapsed_min > SESSION_BUDGET_MIN:
        log.warning(
            "session budget reached (%.0f min); not launching %s. Re-run this "
            "block (or the whole notebook) to continue -- finished experiments "
            "are skipped.",
            elapsed_min,
            exp,
        )
        RUNS.append(
            {
                "exp": exp,
                "status": "not_started",
                "minutes": 0.0,
                "rows": 0,
                "reason": "session budget",
            }
        )
        _save_summary()
        return
    run_experiment(exp)

### B2 -- reconstruction accuracy baseline

In [ ]:
run_block("b2")

### B4 -- frame-budget ablation (headline result)

In [ ]:
run_block("b4")

### Summary

In [ ]:
print()
print("=" * 78)
print(f"{'exp':<5} {'status':<12} {'min':>7} {'rows':>6}  log")
print("-" * 78)
for r in RUNS:
    print(
        f"{r['exp']:<5} {r.get('status', '?'):<12} {r.get('minutes', 0.0):>7.1f} "
        f"{r.get('rows', 0):>6}  {Path(r.get('log', '-')).name}"
    )
print("=" * 78)

failed = [r for r in RUNS if r.get("status") in ("failed", "error", "timeout")]
print(f"\n{len(RUNS) - len(failed)}/{len(RUNS)} ok; summary -> {SUMMARY_PATH}")
for r in failed:
    print(f"\n--- {r['exp']}: {r.get('error', '')} ---")
    print(r.get("tail") or r.get("traceback") or "(see log)")

## 15. Collect results

Everything the paper needs, zipped into one file: the CSVs, every log, and the
run summary. On Kaggle the zip appears under the notebook's Output tab; on Colab
it lands in Drive if you mounted it.

In [ ]:
bundle = OUT_DIR / ("tier_b_" + RUN_ID)
bundle.mkdir(parents=True, exist_ok=True)
(bundle / "results").mkdir(exist_ok=True)

for csv in RESULTS_DIR.glob("*.csv"):
    shutil.copy2(csv, bundle / "results" / csv.name)
shutil.copytree(LOG_DIR, bundle / "logs", dirs_exist_ok=True)
if SUMMARY_PATH.exists():
    shutil.copy2(SUMMARY_PATH, bundle / "run_summary.json")

(bundle / "environment.txt").write_text(
    "\n".join(
        [
            "run_id      " + RUN_ID,
            "platform    " + PLATFORM,
            "commit      " + COMMIT,
            "scope       " + SCOPE,
            "python      " + platform.python_version(),
            "os          " + platform.platform(),
            "curope      " + ("compiled" if CUROPE_BUILT else "pytorch fallback"),
        ]
    ),
    encoding="utf-8",
)

archive = shutil.make_archive(str(bundle), "zip", root_dir=bundle)
log.info("bundle -> %s (%.1f MB)", archive, Path(archive).stat().st_size / 1024**2)

print("\nCSVs produced:")
for csv in sorted((bundle / "results").glob("*.csv")):
    print(f"  {csv.name:<44} {_csv_rows(csv):>6} rows  {csv.stat().st_size / 1024:.0f} KB")

print("\nTo bring these home: download the zip, unpack into bench/results/, then")
print("  git add bench/results && git commit -m 'bench: Tier B results'")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `exit -9 / SIGKILL` | host RAM OOM killer, not VRAM | lower `--n-images`, or `--image-size 384` |
| status `timeout` | exceeded the ceiling section 12 printed — `TIMEOUT_MIN[scope][exp]`, times the scene count in the full lane | raise the per-scene number if the run was genuinely progressing; the log shows where it stalled |
| `capping there so the results survive` | the ceiling was longer than the session budget had left | expected near the end of a session; the experiment resumes on the next run |
| a scan missing from the scene table | its images resolved on no mount and the fetch could not supply them | read the `scan not resolved` warnings; check `DTU_LIGHTING_GLOB` against what the mirror actually ships |
| `does not serve byte ranges` | a mirror or proxy in front of `DTU_IMAGES_URL` / `DTU_POINTS_URL` | attach a Kaggle mirror instead, or point the URL at a server that answers `Range` |
| `torch.cuda.OutOfMemoryError` | VRAM | lower `--image-size` |
| `exit -11 / SIGSEGV` | native crash in VTK or CUDA | read `logs/<run_id>/faulthandler.log` for the Python frames |
| `no scenes` SystemExit | nothing attached, or `DATA_ROOT` is wrong | Kaggle sidebar > Input > Add Input, then point `DATA_ROOT` at the mount |
| dataset test `decode` FAIL | corrupt or unusual bit depth | drop the file, or convert the scene to 8-bit RGB |
| dataset test `ground_truth` FAIL | trimesh cannot open the GT | check it is a real `.ply`/`.obj` and not a Git-LFS pointer or an archive |
| dataset test `tau_vs_gt_scale` WARN | manifest units disagree with the GT file | set `tau`/`units` per scene: 2 mm is DTU's, a COLMAP cloud is in scene units |
| discovery names a scene `Rectified` | the images sit in a folder not in `IMAGE_DIR_NAMES` | add that folder name to `IMAGE_DIR_NAMES`, or list the scene in `SCENES` |
| exits 0, **zero rows** | scene has `gt_path: null`, or `--scenes` matched nothing | scene names must match the manifest exactly |
| B4's three variants identical | scene has <= 40 frames, so the budget never fires | use a real extracted video sequence |
| `MASt3R is not installed` | cell 7 did not finish, or the kernel restarted | re-run cell 7; `--no-deps` means torch is never touched |
| `numpy.dtype size changed` | numpy 2 vs a wheel built for numpy 1 | set `PIN_NUMPY_LT2 = True`, restart the kernel, re-run from cell 2 |
| session killed at 12 h | platform wall clock | re-run; completed CSVs are skipped |
| status `timeout`, **zero rows**, hours of log | the experiment writes its CSV once, when it finishes | nothing partial survives: lower `FULL_SCENE_LIMIT[exp]` or trim `DTU_SCANS` so the run fits a session, then re-run |
| `disk quota exceeded` on Kaggle | reconstructions landing in `/kaggle/working` | confirm `--output-root` points into `/kaggle/temp` |

**For the paper:** run both B2 and B4 on one GPU type. `env_metadata()` stamps
`gpu`, `torch` and `git_commit` onto every row, so mixed hardware is at least
detectable after the fact - but it is not comparable across rows.